# Reasoning Model Use Case Template

This template provides you a simple way to quickly build out production reasoning use cases.

Fill out each of the required sections with your own organization's data, from your use case.

A reference example from a banking credit use case has been included, as placeholder values.

In [21]:
from pydantic import BaseModel, Field
from openai import AzureOpenAI
import os
from dotenv import load_dotenv
import json
import pandas as pd
from tools import execute_tool_function

# Load environment variables
load_dotenv("./.env")

client = AzureOpenAI(  
    api_version="2024-12-01-preview",  
    azure_endpoint=os.getenv("O4MINI_AZURE_ENDPOINT"),  
    api_key=os.getenv("O4MINI_API_KEY")  
)  


def o4minicall(prompt, reasoning_effort, tools=None, tool_choice="auto", response_format=None):
    """
    Call the O4 Mini model with optional tool calling functionality.
    
    Args:
        prompt (str): The user's query or prompt
        reasoning_effort (str): The reasoning effort level (low, medium, high)
        tools (list, optional): List of tool definitions for function calling
        tool_choice (str or dict, optional): Tool selection strategy
        response_format (dict, optional): Structured output format
        
    Returns:
        dict or str: The model's response, potentially including tool results
    """
    system_message = """
    You are a helpful AI assistant that reasons overs a fresh production plan for perishable food and analyzes potential contributos of waste based on fresh production, markdown and sales data.
    """
    
    # Create the base parameters for the API call
    params = {
        "model": "o4-mini",
        "messages": [
            {
                "role": "user", 
                "content": system_message + " " + prompt
            }
        ],
        "reasoning_effort": reasoning_effort
    }
    
    # Add tools if provided
    if tools is not None:
        params["tools"] = tools
        params["tool_choice"] = tool_choice
    
    # Make the initial API call
    response = client.chat.completions.create(**params)
    response_message = response.choices[0].message
    
    # Check if the model wants to call a function
    if hasattr(response_message, 'tool_calls') and response_message.tool_calls:
        # Process function calls and get results
        tool_results = []
        
        for tool_call in response_message.tool_calls:
            # Extract function call details
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            # Execute the function
            result = execute_tool_function(function_name, function_args)
            
            tool_results.append({
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result)
            })
        
        # Format the datasets from tool calls
        datasets_from_tool_calls = ""
        for result in tool_results:
            datasets_from_tool_calls += f"\n### Results from {result['name']}:\n"
            datasets_from_tool_calls += f"```json\n{result['content']}\n```\n"
        
        # Generate a new prompt with the tool call results incorporated
        new_prompt = f"""
        {prompt}
        
        Here is additional data retrieved to help with your analysis:
        {datasets_from_tool_calls}
        
        Please use this data to enhance your analysis.
        """
        
        # Create parameters for the second call with the new prompt
        second_params = {
            "model": "o4-mini",
            "messages": [
                {
                    "role": "user", 
                    "content": system_message + " " + new_prompt
                }
            ],
            "reasoning_effort": reasoning_effort
        }
        
        # Add response_format if provided
        if response_format is not None:
            second_params["response_format"] = response_format
            completion = client.beta.chat.completions.parse(**second_params)
            if hasattr(completion.choices[0].message, 'parsed'):
                return completion.choices[0].message.parsed
            else:
                return completion.choices[0].message.content
        else:
            completion = client.chat.completions.create(**second_params)
            return completion.choices[0].message.content
    
    # Handle the case where no function call is made
    if response_format is not None:
        # Switch to using parse for structured output
        parse_params = params.copy()
        parse_params["response_format"] = response_format
        completion = client.beta.chat.completions.parse(**parse_params)
        if hasattr(completion.choices[0].message, 'parsed'):
            return completion.choices[0].message.parsed
        else:
            return completion.choices[0].message.content
    else:
        return response_message.content


## Define inputs

### Instructions and task (required)

The instructions for what the model needs to do, and the specific input task.

In [22]:
instructions_and_task="""

"Given these datasets: #Dataset-1 [Fresh Production Plan for Rotisserie Chickens], #Dataset-2 [Markdowns] and #Dataset-3 [Sales history] for Store# 5260 determine if there are any statistically significant correlations that lead to overproduction, underproduction, increased waste or lost sales. Use these columns from their respective datasets. shift timings (columns: shift_start_tme
, shift_end_time), production plan forecast (column: plan_production_qty), actual production (column: plan_actual_production_qty), throwaways/waste (column: MUMD_QTY) and actual sales (column: sales_unit_qty). Explain your reasoning."

Provide a detailed explanation of your analyses and recommendations. Include all intermediate calculations, evidence from the datasets, and a discussion on which factors most significantly contributed to your analyses. Be as elaborate as needed and make the explanation easily understandable.

---

### <format>

Please include a brief summary of the fresh production plan for 4/21/2025 as-is from the Fresh Production dataset at the top of your response. Then highlight your key findings and recommendations. Provide detailed intermediate calculations, reference specific datasets as evidence, and present your insights with a clear explanation of each step using markdown formatting. All working out must be shown, and any tables must be fully filled out. Conclude with a detailed agenda for the store manager to discuss with their associates based on your analyses.

---

"""

In [23]:
# Define optional variables here- if they are commented out in lower sections, they will simply be blank strings.
few_shot_examples=""
provided_rules_and_policies=""


### Datasets (required)

The datasets needed to solve the problem.

Add your own datasets for the model to reason over.

Here, the data is included directly into the prompt as static text. In a production application, you will want to dynamically pull each data table specific to the input task/target customer, etc. This could involve a SQL query or Azure AI Search query.

An example of how this can be done is shown in the optional Function Calling section.

In [24]:
datasets="""
Dataset 1: Fresh Production Plan for Rotisserie Chickens
*This dataset is represented as a table showing the rotisseri chicken fresh production plan (column: plan_production_qty), actual production (column: plan_actual_production_qty)


|store_nbr|country_code|timezone_txt|utc_offset_seconds|plan_date|product_upc_nbr|product_name|product_production_method_id|product_production_method_name|product_area_id|product_area_name|product_hold_tme|product_production_tme|shift_id|shift_start_tme|shift_end_tme|forecast_demand_qty|forecast_production_safety_stock_qty|plan_generation_start_ts|plan_generation_complete_ts|plan_on_hand_qty|plan_max_on_hand_alert_qty|plan_carry_over_qty|plan_min_presentation_qty|plan_actual_production_qty|plan_actual_breakout_qty|plan_production_qty|plan_production_man_adjusted_qty|plan_sales_yesterday_applied_qty|plan_throws_yesterday_applied_qty|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|5260|US|CST|-21600|2025-02-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|77.51||2025-02-10 06:30:45.000000 UTC|2025-02-10 06:39:32.000000 UTC|155||0|28|60|0|30||0|0|
|5260|US|CST|-21600|2025-02-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|77.51||2025-02-10 06:30:45.000000 UTC|2025-02-10 06:39:32.000000 UTC|155||0|28|21|0|21||0|0|
|5260|US|CST|-21600|2025-02-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|77.51||2025-02-10 06:30:45.000000 UTC|2025-02-10 06:39:32.000000 UTC|155||0|28|21|0|21||0|0|
|5260|US|CST|-21600|2025-02-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|77.51||2025-02-10 06:30:45.000000 UTC|2025-02-10 06:39:32.000000 UTC|155||0|28|21|0|21||0|0|
|5260|US|CDT|-18000|2025-03-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|81.6||2025-03-15 05:30:46.000000 UTC|2025-03-15 05:38:49.000000 UTC|45||0|26|22|0|22||0|0|
|5260|US|CDT|-18000|2025-03-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|81.6||2025-03-15 05:30:46.000000 UTC|2025-03-15 05:38:49.000000 UTC|45||0|26|28|0|28||0|0|
|5260|US|CDT|-18000|2025-03-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|81.6||2025-03-15 05:30:46.000000 UTC|2025-03-15 05:38:49.000000 UTC|45||0|26|30|0|30||0|0|
|5260|US|CDT|-18000|2025-03-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|81.6||2025-03-15 05:30:46.000000 UTC|2025-03-15 05:38:49.000000 UTC|45||0|26|22|0|22||0|0|
|5260|US|CDT|-18000|2025-03-24|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|94.62||2025-03-24 05:33:05.000000 UTC|2025-03-24 05:41:53.000000 UTC|148||0|26|40|0|40||0|0|
|5260|US|CDT|-18000|2025-03-24|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|94.62||2025-03-24 05:33:05.000000 UTC|2025-03-24 05:41:53.000000 UTC|148||0|26|18|0|32||0|0|
|5260|US|CDT|-18000|2025-03-24|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|94.62||2025-03-24 05:33:05.000000 UTC|2025-03-24 05:41:53.000000 UTC|148||0|26|17|0|17||0|0|
|5260|US|CDT|-18000|2025-03-24|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|94.62||2025-03-24 05:33:05.000000 UTC|2025-03-24 05:41:53.000000 UTC|148||0|26|0|0|23||0|0|
|5260|US|CST|-21600|2025-02-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|85.79||2025-02-03 06:33:00.000000 UTC|2025-02-03 06:41:42.000000 UTC|178||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|85.79||2025-02-03 06:33:00.000000 UTC|2025-02-03 06:41:42.000000 UTC|178||0|28|0|0|22||0|0|
|5260|US|CST|-21600|2025-02-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|85.79||2025-02-03 06:33:00.000000 UTC|2025-02-03 06:41:42.000000 UTC|178||0|28|32|0|32||0|0|
|5260|US|CST|-21600|2025-02-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|85.79||2025-02-03 06:33:00.000000 UTC|2025-02-03 06:41:42.000000 UTC|178||0|28|29|0|29||0|0|
|5260|US|CST|-21600|2025-02-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|57.18||2025-02-19 06:31:03.000000 UTC|2025-02-19 06:39:52.000000 UTC|0||0|28|0|0|15||0|0|
|5260|US|CST|-21600|2025-02-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|57.18||2025-02-19 06:31:03.000000 UTC|2025-02-19 06:39:52.000000 UTC|0||0|28|0|0|15||0|0|
|5260|US|CST|-21600|2025-02-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|57.18||2025-02-19 06:31:03.000000 UTC|2025-02-19 06:39:52.000000 UTC|0||0|28|0|0|19||0|0|
|5260|US|CST|-21600|2025-02-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|57.18||2025-02-19 06:31:03.000000 UTC|2025-02-19 06:39:52.000000 UTC|0||0|28|0|0|26||0|0|
|5260|US|CDT|-18000|2025-03-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|71.94||2025-03-14 05:33:04.000000 UTC|2025-03-14 05:41:08.000000 UTC|163||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-03-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|71.94||2025-03-14 05:33:04.000000 UTC|2025-03-14 05:41:08.000000 UTC|163||0|26|26|0|26||0|0|
|5260|US|CDT|-18000|2025-03-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|71.94||2025-03-14 05:33:04.000000 UTC|2025-03-14 05:41:08.000000 UTC|163||0|26|0|0|16||0|0|
|5260|US|CDT|-18000|2025-03-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|71.94||2025-03-14 05:33:04.000000 UTC|2025-03-14 05:41:08.000000 UTC|163||0|26|26|0|26||0|0|
|5260|US|CDT|-18000|2025-04-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|61.76||2025-04-04 05:32:09.000000 UTC|2025-04-04 05:40:27.000000 UTC|117||0|26|0|0|13||0|0|
|5260|US|CDT|-18000|2025-04-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|61.76||2025-04-04 05:32:09.000000 UTC|2025-04-04 05:40:27.000000 UTC|117||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-04-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|61.76||2025-04-04 05:32:09.000000 UTC|2025-04-04 05:40:27.000000 UTC|117||0|26|17|0|17||0|0|
|5260|US|CDT|-18000|2025-04-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|61.76||2025-04-04 05:32:09.000000 UTC|2025-04-04 05:40:27.000000 UTC|117||0|26|27|0|27||0|0|
|5260|US|CDT|-18000|2025-03-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|69.42||2025-03-19 05:31:06.000000 UTC|2025-03-19 05:39:49.000000 UTC|54||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-03-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|69.42||2025-03-19 05:31:06.000000 UTC|2025-03-19 05:39:49.000000 UTC|54||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-03-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|69.42||2025-03-19 05:31:06.000000 UTC|2025-03-19 05:39:49.000000 UTC|54||0|26|28|0|28||0|0|
|5260|US|CDT|-18000|2025-03-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|69.42||2025-03-19 05:31:06.000000 UTC|2025-03-19 05:39:49.000000 UTC|54||0|26|0|0|23||0|0|
|5260|US|CST|-21600|2025-03-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|85.09||2025-03-03 06:31:20.000000 UTC|2025-03-03 06:39:42.000000 UTC|57||0|26|44|0|22||0|0|
|5260|US|CST|-21600|2025-03-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|85.09||2025-03-03 06:31:20.000000 UTC|2025-03-03 06:39:42.000000 UTC|57||0|26|0|0|22||0|0|
|5260|US|CST|-21600|2025-03-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|85.09||2025-03-03 06:31:20.000000 UTC|2025-03-03 06:39:42.000000 UTC|57||0|26|41|0|30||0|0|
|5260|US|CST|-21600|2025-03-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|85.09||2025-03-03 06:31:20.000000 UTC|2025-03-03 06:39:42.000000 UTC|57||0|26|28|0|28||0|0|
|5260|US|CDT|-18000|2025-04-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|59.48||2025-04-08 05:32:08.000000 UTC|2025-04-08 05:40:25.000000 UTC|65||0|26|14|0|14||0|0|
|5260|US|CDT|-18000|2025-04-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|59.48||2025-04-08 05:32:08.000000 UTC|2025-04-08 05:40:25.000000 UTC|65||0|26|0|0|14||0|0|
|5260|US|CDT|-18000|2025-04-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|59.48||2025-04-08 05:32:08.000000 UTC|2025-04-08 05:40:25.000000 UTC|65||0|26|24|0|24||0|0|
|5260|US|CDT|-18000|2025-04-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|59.48||2025-04-08 05:32:08.000000 UTC|2025-04-08 05:40:25.000000 UTC|65||0|26|23|0|23||0|0|
|5260|US|CDT|-18000|2025-03-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|84.47||2025-03-10 05:32:14.000000 UTC|2025-03-10 05:40:25.000000 UTC|72||0|26|20|0|20||0|0|
|5260|US|CDT|-18000|2025-03-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|84.47||2025-03-10 05:32:14.000000 UTC|2025-03-10 05:40:25.000000 UTC|72||0|26|0|0|26||0|0|
|5260|US|CDT|-18000|2025-03-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|84.47||2025-03-10 05:32:14.000000 UTC|2025-03-10 05:40:25.000000 UTC|72||0|26|20|0|20||0|0|
|5260|US|CDT|-18000|2025-03-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|84.47||2025-03-10 05:32:14.000000 UTC|2025-03-10 05:40:25.000000 UTC|72||0|26|13|0|35||0|0|
|5260|US|CST|-21600|2025-03-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|85.08||2025-03-02 06:31:45.000000 UTC|2025-03-02 06:40:10.000000 UTC|45||0|26|28|0|28||0|0|
|5260|US|CST|-21600|2025-03-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|85.08||2025-03-02 06:31:45.000000 UTC|2025-03-02 06:40:10.000000 UTC|45||0|26|60|0|21||0|0|
|5260|US|CST|-21600|2025-03-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|85.08||2025-03-02 06:31:45.000000 UTC|2025-03-02 06:40:10.000000 UTC|45||0|26|31|0|31||0|0|
|5260|US|CST|-21600|2025-03-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|85.08||2025-03-02 06:31:45.000000 UTC|2025-03-02 06:40:10.000000 UTC|45||0|26|21|0|21||0|0|
|5260|US|CST|-21600|2025-02-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|76.42||2025-02-15 06:32:30.000000 UTC|2025-02-15 06:41:22.000000 UTC|49||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|76.42||2025-02-15 06:32:30.000000 UTC|2025-02-15 06:41:22.000000 UTC|49||0|28|17|0|17||0|0|
|5260|US|CST|-21600|2025-02-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|76.42||2025-02-15 06:32:30.000000 UTC|2025-02-15 06:41:22.000000 UTC|49||0|28|28|0|28||0|0|
|5260|US|CST|-21600|2025-02-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|76.42||2025-02-15 06:32:30.000000 UTC|2025-02-15 06:41:22.000000 UTC|49||0|28|30|0|30||0|0|
|5260|US|CST|-21600|2025-02-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|75.32||2025-02-11 06:32:48.000000 UTC|2025-02-11 06:41:35.000000 UTC|167||0|28|26|0|26||0|0|
|5260|US|CST|-21600|2025-02-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|75.32||2025-02-11 06:32:48.000000 UTC|2025-02-11 06:41:35.000000 UTC|167||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|75.32||2025-02-11 06:32:48.000000 UTC|2025-02-11 06:41:35.000000 UTC|167||0|28|0|0|22||0|0|
|5260|US|CST|-21600|2025-02-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|75.32||2025-02-11 06:32:48.000000 UTC|2025-02-11 06:41:35.000000 UTC|167||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|74.78||2025-02-12 06:30:44.000000 UTC|2025-02-12 06:39:37.000000 UTC|182||0|28|0|0|15||0|0|
|5260|US|CST|-21600|2025-02-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|74.78||2025-02-12 06:30:44.000000 UTC|2025-02-12 06:39:37.000000 UTC|182||0|28|44|0|22||0|0|
|5260|US|CST|-21600|2025-02-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|74.78||2025-02-12 06:30:44.000000 UTC|2025-02-12 06:39:37.000000 UTC|182||0|28|0|0|33||0|0|
|5260|US|CST|-21600|2025-02-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|74.78||2025-02-12 06:30:44.000000 UTC|2025-02-12 06:39:37.000000 UTC|182||0|28|22|0|22||0|0|
|5260|US|CDT|-18000|2025-03-31|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|75.73||2025-03-31 05:30:58.000000 UTC|2025-03-31 05:39:11.000000 UTC|100||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-03-31|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|75.73||2025-03-31 05:30:58.000000 UTC|2025-03-31 05:39:11.000000 UTC|100||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-03-31|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|75.73||2025-03-31 05:30:58.000000 UTC|2025-03-31 05:39:11.000000 UTC|100||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-03-31|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|75.73||2025-03-31 05:30:58.000000 UTC|2025-03-31 05:39:11.000000 UTC|100||0|26|36|0|30||0|0|
|5260|US|CST|-21600|2025-02-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|76.41||2025-02-14 06:30:24.000000 UTC|2025-02-14 06:39:00.000000 UTC|132||0|28|19|0|19||0|0|
|5260|US|CST|-21600|2025-02-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|76.41||2025-02-14 06:30:24.000000 UTC|2025-02-14 06:39:00.000000 UTC|132||0|28|25|0|25||0|0|
|5260|US|CST|-21600|2025-02-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|76.41||2025-02-14 06:30:24.000000 UTC|2025-02-14 06:39:00.000000 UTC|132||0|28|28|0|28||0|0|
|5260|US|CST|-21600|2025-02-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|76.41||2025-02-14 06:30:24.000000 UTC|2025-02-14 06:39:00.000000 UTC|132||0|28|19|0|19||0|0|
|5260|US|CDT|-18000|2025-04-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|67.29||2025-04-05 05:30:32.000000 UTC|2025-04-05 05:38:55.000000 UTC|33||0|26|32|0|32||0|0|
|5260|US|CDT|-18000|2025-04-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|67.29||2025-04-05 05:30:32.000000 UTC|2025-04-05 05:38:55.000000 UTC|33||0|26|0|0|18||0|0|
|5260|US|CDT|-18000|2025-04-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|67.29||2025-04-05 05:30:32.000000 UTC|2025-04-05 05:38:55.000000 UTC|33||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-04-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|67.29||2025-04-05 05:30:32.000000 UTC|2025-04-05 05:38:55.000000 UTC|33||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-04-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|80.37||2025-04-06 05:30:38.000000 UTC|2025-04-06 05:38:45.000000 UTC|41||0|26|31|0|31||0|0|
|5260|US|CDT|-18000|2025-04-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|80.37||2025-04-06 05:30:38.000000 UTC|2025-04-06 05:38:45.000000 UTC|41||0|26|27|0|27||0|0|
|5260|US|CDT|-18000|2025-04-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|80.37||2025-04-06 05:30:38.000000 UTC|2025-04-06 05:38:45.000000 UTC|41||0|26|20|0|20||0|0|
|5260|US|CDT|-18000|2025-04-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|80.37||2025-04-06 05:30:38.000000 UTC|2025-04-06 05:38:45.000000 UTC|41||0|26|0|0|20||0|0|
|5260|US|CDT|-18000|2025-04-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|80.38||2025-04-19 05:30:28.000000 UTC|2025-04-19 05:38:30.000000 UTC|189||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-04-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|80.38||2025-04-19 05:30:28.000000 UTC|2025-04-19 05:38:30.000000 UTC|189||0|26|29|0|29||0|0|
|5260|US|CDT|-18000|2025-04-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|80.38||2025-04-19 05:30:28.000000 UTC|2025-04-19 05:38:30.000000 UTC|189||0|26|33|0|33||0|0|
|5260|US|CDT|-18000|2025-04-19|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|80.38||2025-04-19 05:30:28.000000 UTC|2025-04-19 05:38:30.000000 UTC|189||0|26|0|0|23||0|0|
|5260|US|CDT|-18000|2025-03-27|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|81.19||2025-03-27 05:32:49.000000 UTC|2025-03-27 05:41:15.000000 UTC|130||0|26|22|0|22||0|0|
|5260|US|CDT|-18000|2025-03-27|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|81.19||2025-03-27 05:32:49.000000 UTC|2025-03-27 05:41:15.000000 UTC|130||0|26|0|0|34||0|0|
|5260|US|CDT|-18000|2025-03-27|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|81.19||2025-03-27 05:32:49.000000 UTC|2025-03-27 05:41:15.000000 UTC|130||0|26|22|0|22||0|0|
|5260|US|CDT|-18000|2025-03-27|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|81.19||2025-03-27 05:32:49.000000 UTC|2025-03-27 05:41:15.000000 UTC|130||0|26|0|0|22||0|0|
|5260|US|CDT|-18000|2025-04-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|59.48||2025-04-11 05:30:31.000000 UTC|2025-04-11 05:38:34.000000 UTC|109||0|26|16|0|16||0|0|
|5260|US|CDT|-18000|2025-04-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|59.48||2025-04-11 05:30:31.000000 UTC|2025-04-11 05:38:34.000000 UTC|109||0|26|0|0|16||0|0|
|5260|US|CDT|-18000|2025-04-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|59.48||2025-04-11 05:30:31.000000 UTC|2025-04-11 05:38:34.000000 UTC|109||0|26|27|0|27||0|0|
|5260|US|CDT|-18000|2025-04-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|59.48||2025-04-11 05:30:31.000000 UTC|2025-04-11 05:38:34.000000 UTC|109||0|26|16|0|16||0|0|
|5260|US|CDT|-18000|2025-03-23|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|94.45||2025-03-23 05:32:17.000000 UTC|2025-03-23 05:40:34.000000 UTC|70||0|26|26|0|26||0|0|
|5260|US|CDT|-18000|2025-03-23|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|94.45||2025-03-23 05:32:17.000000 UTC|2025-03-23 05:40:34.000000 UTC|70||0|26|26|0|26||0|0|
|5260|US|CDT|-18000|2025-03-23|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|94.45||2025-03-23 05:32:17.000000 UTC|2025-03-23 05:40:34.000000 UTC|70||0|26|35|0|35||0|0|
|5260|US|CDT|-18000|2025-03-23|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|94.45||2025-03-23 05:32:17.000000 UTC|2025-03-23 05:40:34.000000 UTC|70||0|26|0|0|26||0|0|
|5260|US|CST|-21600|2025-02-26|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|68.67||2025-02-26 06:30:34.000000 UTC|2025-02-26 06:38:43.000000 UTC|92||0|28|27|0|27||0|0|
|5260|US|CST|-21600|2025-02-26|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|68.67||2025-02-26 06:30:34.000000 UTC|2025-02-26 06:38:43.000000 UTC|92||0|28|23|0|23||0|0|
|5260|US|CST|-21600|2025-02-26|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|68.67||2025-02-26 06:30:34.000000 UTC|2025-02-26 06:38:43.000000 UTC|92||0|28|17|0|17||0|0|
|5260|US|CST|-21600|2025-02-26|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|68.67||2025-02-26 06:30:34.000000 UTC|2025-02-26 06:38:43.000000 UTC|92||0|28|0|0|17||0|0|
|5260|US|CDT|-18000|2025-03-28|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|80.58||2025-03-28 05:31:09.000000 UTC|2025-03-28 05:39:35.000000 UTC|118||0|26|0|0|22||0|0|
|5260|US|CDT|-18000|2025-03-28|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|80.58||2025-03-28 05:31:09.000000 UTC|2025-03-28 05:39:35.000000 UTC|118||0|26|32|0|32||0|0|
|5260|US|CDT|-18000|2025-03-28|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|80.58||2025-03-28 05:31:09.000000 UTC|2025-03-28 05:39:35.000000 UTC|118||0|26|22|0|22||0|0|
|5260|US|CDT|-18000|2025-03-28|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|80.58||2025-03-28 05:31:09.000000 UTC|2025-03-28 05:39:35.000000 UTC|118||0|26|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|84.71||2025-02-09 06:33:00.000000 UTC|2025-02-09 06:41:31.000000 UTC|101||0|28|62|0|31||0|0|
|5260|US|CST|-21600|2025-02-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|84.71||2025-02-09 06:33:00.000000 UTC|2025-02-09 06:41:31.000000 UTC|101||0|28|30|0|30||0|0|
|5260|US|CST|-21600|2025-02-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|84.71||2025-02-09 06:33:00.000000 UTC|2025-02-09 06:41:31.000000 UTC|101||0|28|19|0|19||0|0|
|5260|US|CST|-21600|2025-02-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|84.71||2025-02-09 06:33:00.000000 UTC|2025-02-09 06:41:31.000000 UTC|101||0|28|25|0|25||0|0|
|5260|US|CST|-21600|2025-02-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|69.24||2025-02-18 06:31:07.000000 UTC|2025-02-18 06:39:51.000000 UTC|14||0|28|0|0|19||0|0|
|5260|US|CST|-21600|2025-02-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|69.24||2025-02-18 06:31:07.000000 UTC|2025-02-18 06:39:51.000000 UTC|14||0|28|0|0|19||0|0|
|5260|US|CST|-21600|2025-02-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|69.24||2025-02-18 06:31:07.000000 UTC|2025-02-18 06:39:51.000000 UTC|14||0|28|0|0|19||0|0|
|5260|US|CST|-21600|2025-02-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|69.24||2025-02-18 06:31:07.000000 UTC|2025-02-18 06:39:51.000000 UTC|14||0|28|28|0|28||0|0|
|5260|US|CST|-21600|2025-03-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|72||2025-03-06 06:30:28.000000 UTC|2025-03-06 06:39:25.000000 UTC|78||0|26|51|0|17||0|0|
|5260|US|CST|-21600|2025-03-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|72||2025-03-06 06:30:28.000000 UTC|2025-03-06 06:39:25.000000 UTC|78||0|26|27|0|27||0|0|
|5260|US|CST|-21600|2025-03-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|72||2025-03-06 06:30:28.000000 UTC|2025-03-06 06:39:25.000000 UTC|78||0|26|22|0|22||0|0|
|5260|US|CST|-21600|2025-03-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|72||2025-03-06 06:30:28.000000 UTC|2025-03-06 06:39:25.000000 UTC|78||0|26|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|73.62||2025-02-06 06:32:37.000000 UTC|2025-02-06 06:41:15.000000 UTC|73||0|28|20|0|20||0|0|
|5260|US|CST|-21600|2025-02-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|73.62||2025-02-06 06:32:37.000000 UTC|2025-02-06 06:41:15.000000 UTC|73||0|28|0|0|20||0|0|
|5260|US|CST|-21600|2025-02-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|73.62||2025-02-06 06:32:37.000000 UTC|2025-02-06 06:41:15.000000 UTC|73||0|28|0|0|33||0|0|
|5260|US|CST|-21600|2025-02-06|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|73.62||2025-02-06 06:32:37.000000 UTC|2025-02-06 06:41:15.000000 UTC|73||0|28|20|0|20||0|0|
|5260|US|CST|-21600|2025-02-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|88.01||2025-02-01 06:30:30.000000 UTC|2025-02-01 06:39:32.000000 UTC|115||0|28|23|0|23||0|0|
|5260|US|CST|-21600|2025-02-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|88.01||2025-02-01 06:30:30.000000 UTC|2025-02-01 06:39:32.000000 UTC|115||0|28|23|0|23||0|0|
|5260|US|CST|-21600|2025-02-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|88.01||2025-02-01 06:30:30.000000 UTC|2025-02-01 06:39:32.000000 UTC|115||0|28|36|0|36||0|0|
|5260|US|CST|-21600|2025-02-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|88.01||2025-02-01 06:30:30.000000 UTC|2025-02-01 06:39:32.000000 UTC|115||0|28|0|0|28||0|0|
|5260|US|CST|-21600|2025-02-25|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|67.68||2025-02-25 06:32:38.000000 UTC|2025-02-25 06:41:14.000000 UTC|158||0|28|24|0|24||0|0|
|5260|US|CST|-21600|2025-02-25|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|67.68||2025-02-25 06:32:38.000000 UTC|2025-02-25 06:41:14.000000 UTC|158||0|28|21|0|21||0|0|
|5260|US|CST|-21600|2025-02-25|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|67.68||2025-02-25 06:32:38.000000 UTC|2025-02-25 06:41:14.000000 UTC|158||0|28|0|0|16||0|0|
|5260|US|CST|-21600|2025-02-25|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|67.68||2025-02-25 06:32:38.000000 UTC|2025-02-25 06:41:14.000000 UTC|158||0|28|21|0|21||0|0|
|5260|US|CST|-21600|2025-03-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|88.12||2025-03-09 06:30:51.000000 UTC|2025-03-09 06:39:37.000000 UTC|209||0|26|0|0|27||0|0|
|5260|US|CST|-21600|2025-03-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|88.12||2025-03-09 06:30:51.000000 UTC|2025-03-09 06:39:37.000000 UTC|209||0|26|21|0|21||0|0|
|5260|US|CST|-21600|2025-03-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|88.12||2025-03-09 06:30:51.000000 UTC|2025-03-09 06:39:37.000000 UTC|209||0|26|21|0|21||0|0|
|5260|US|CST|-21600|2025-03-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|88.12||2025-03-09 06:30:51.000000 UTC|2025-03-09 06:39:37.000000 UTC|209||0|26|36|0|36||0|0|
|5260|US|CST|-21600|2025-02-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|80.81||2025-02-16 06:30:47.000000 UTC|2025-02-16 06:40:22.000000 UTC|254||0|28|38|0|38||0|0|
|5260|US|CST|-21600|2025-02-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|80.81||2025-02-16 06:30:47.000000 UTC|2025-02-16 06:40:22.000000 UTC|254||0|28|17|0|17||0|0|
|5260|US|CST|-21600|2025-02-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|80.81||2025-02-16 06:30:47.000000 UTC|2025-02-16 06:40:22.000000 UTC|254||0|28|0|0|22||0|0|
|5260|US|CST|-21600|2025-02-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|80.81||2025-02-16 06:30:47.000000 UTC|2025-02-16 06:40:22.000000 UTC|254||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|69.68||2025-02-17 06:30:25.000000 UTC|2025-02-17 06:39:23.000000 UTC|117||0|28|32|0|16||0|0|
|5260|US|CST|-21600|2025-02-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|69.68||2025-02-17 06:30:25.000000 UTC|2025-02-17 06:39:23.000000 UTC|117||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|69.68||2025-02-17 06:30:25.000000 UTC|2025-02-17 06:39:23.000000 UTC|117||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|69.68||2025-02-17 06:30:25.000000 UTC|2025-02-17 06:39:23.000000 UTC|117||0|28|25|0|25||0|0|
|5260|US|CST|-21600|2025-02-23|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|84.3||2025-02-23 06:31:59.000000 UTC|2025-02-23 06:40:16.000000 UTC|8||0|28|49|0|21||0|0|
|5260|US|CST|-21600|2025-02-23|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|84.3||2025-02-23 06:31:59.000000 UTC|2025-02-23 06:40:16.000000 UTC|8||0|28|21|0|21||0|0|
|5260|US|CST|-21600|2025-02-23|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|84.3||2025-02-23 06:31:59.000000 UTC|2025-02-23 06:40:16.000000 UTC|8||0|28|21|0|21||0|0|
|5260|US|CST|-21600|2025-02-23|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|84.3||2025-02-23 06:31:59.000000 UTC|2025-02-23 06:40:16.000000 UTC|8||0|28|38|0|38||0|0|
|5260|US|CST|-21600|2025-02-24|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|70.64||2025-02-24 06:30:36.000000 UTC|2025-02-24 06:39:03.000000 UTC|0||0|28|19|0|19||0|0|
|5260|US|CST|-21600|2025-02-24|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|70.64||2025-02-24 06:30:36.000000 UTC|2025-02-24 06:39:03.000000 UTC|0||0|28|19|0|19||0|0|
|5260|US|CST|-21600|2025-02-24|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|70.64||2025-02-24 06:30:36.000000 UTC|2025-02-24 06:39:03.000000 UTC|0||0|28|0|0|19||0|0|
|5260|US|CST|-21600|2025-02-24|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|70.64||2025-02-24 06:30:36.000000 UTC|2025-02-24 06:39:03.000000 UTC|0||0|28|27|0|27||0|0|
|5260|US|CST|-21600|2025-02-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|75.32||2025-02-13 06:31:02.000000 UTC|2025-02-13 06:39:40.000000 UTC|175||0|28|0|0|33||0|0|
|5260|US|CST|-21600|2025-02-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|75.32||2025-02-13 06:31:02.000000 UTC|2025-02-13 06:39:40.000000 UTC|175||0|28|21|0|21||0|0|
|5260|US|CST|-21600|2025-02-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|75.32||2025-02-13 06:31:02.000000 UTC|2025-02-13 06:39:40.000000 UTC|175||0|28|21|0|21||0|0|
|5260|US|CST|-21600|2025-02-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|75.32||2025-02-13 06:31:02.000000 UTC|2025-02-13 06:39:40.000000 UTC|175||0|28|21|0|21||0|0|
|5260|US|CDT|-18000|2025-04-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|66.58||2025-04-07 05:30:39.000000 UTC|2025-04-07 05:39:03.000000 UTC|156||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-04-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|66.58||2025-04-07 05:30:39.000000 UTC|2025-04-07 05:39:03.000000 UTC|156||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-04-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|66.58||2025-04-07 05:30:39.000000 UTC|2025-04-07 05:39:03.000000 UTC|156||0|26|0|0|18||0|0|
|5260|US|CDT|-18000|2025-04-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|66.58||2025-04-07 05:30:39.000000 UTC|2025-04-07 05:39:03.000000 UTC|156||0|26|21|0|28||0|0|
|5260|US|CDT|-18000|2025-03-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|70.99||2025-03-18 05:30:23.000000 UTC|2025-03-18 05:38:50.000000 UTC|37||0|26|16|0|16||0|0|
|5260|US|CDT|-18000|2025-03-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|70.99||2025-03-18 05:30:23.000000 UTC|2025-03-18 05:38:50.000000 UTC|37||0|26|26|0|26||0|0|
|5260|US|CDT|-18000|2025-03-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|70.99||2025-03-18 05:30:23.000000 UTC|2025-03-18 05:38:50.000000 UTC|37||0|26|29|0|29||0|0|
|5260|US|CDT|-18000|2025-03-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|70.99||2025-03-18 05:30:23.000000 UTC|2025-03-18 05:38:50.000000 UTC|37||0|26|0|0|16||0|0|
|5260|US|CST|-21600|2025-02-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|92.65||2025-02-02 06:31:04.000000 UTC|2025-02-02 06:40:47.000000 UTC|101||0|28|41|0|41||0|0|
|5260|US|CST|-21600|2025-02-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|92.65||2025-02-02 06:31:04.000000 UTC|2025-02-02 06:40:47.000000 UTC|101||0|28|23|0|23||0|0|
|5260|US|CST|-21600|2025-02-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|92.65||2025-02-02 06:31:04.000000 UTC|2025-02-02 06:40:47.000000 UTC|101||0|28|23|0|23||0|0|
|5260|US|CST|-21600|2025-02-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|92.65||2025-02-02 06:31:04.000000 UTC|2025-02-02 06:40:47.000000 UTC|101||0|28|23|0|23||0|0|
|5260|US|CST|-21600|2025-02-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|74.17||2025-02-04 06:32:47.000000 UTC|2025-02-04 06:42:12.000000 UTC|109||0|28|29|0|29||0|0|
|5260|US|CST|-21600|2025-02-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|74.17||2025-02-04 06:32:47.000000 UTC|2025-02-04 06:42:12.000000 UTC|109||0|28|19|0|19||0|0|
|5260|US|CST|-21600|2025-02-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|74.17||2025-02-04 06:32:47.000000 UTC|2025-02-04 06:42:12.000000 UTC|109||0|28|19|0|19||0|0|
|5260|US|CST|-21600|2025-02-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|74.17||2025-02-04 06:32:47.000000 UTC|2025-02-04 06:42:12.000000 UTC|109||0|28|15|0|25||0|0|
|5260|US|CST|-21600|2025-02-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|73.06||2025-02-05 06:30:59.000000 UTC|2025-02-05 06:39:59.000000 UTC|104||0|28|0|0|10||0|0|
|5260|US|CST|-21600|2025-02-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|73.06||2025-02-05 06:30:59.000000 UTC|2025-02-05 06:39:59.000000 UTC|104||0|28|30|0|30||0|0|
|5260|US|CST|-21600|2025-02-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|73.06||2025-02-05 06:30:59.000000 UTC|2025-02-05 06:39:59.000000 UTC|104||0|28|20|0|20||0|0|
|5260|US|CST|-21600|2025-02-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|73.06||2025-02-05 06:30:59.000000 UTC|2025-02-05 06:39:59.000000 UTC|104||0|28|32|0|32||0|0|
|5260|US|CDT|-18000|2025-03-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|74.12||2025-03-11 05:31:58.000000 UTC|2025-03-11 05:40:30.000000 UTC|45||0|26|16|0|16||0|0|
|5260|US|CDT|-18000|2025-03-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|74.12||2025-03-11 05:31:58.000000 UTC|2025-03-11 05:40:30.000000 UTC|45||0|26|0|0|27||0|0|
|5260|US|CDT|-18000|2025-03-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|74.12||2025-03-11 05:31:58.000000 UTC|2025-03-11 05:40:30.000000 UTC|45||0|26|0|0|16||0|0|
|5260|US|CDT|-18000|2025-03-11|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|74.12||2025-03-11 05:31:58.000000 UTC|2025-03-11 05:40:30.000000 UTC|45||0|26|31|0|31||0|0|
|5260|US|CDT|-18000|2025-03-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|72.48||2025-03-12 05:32:25.000000 UTC|2025-03-12 05:40:59.000000 UTC|86||0|26|0|0|24||0|0|
|5260|US|CDT|-18000|2025-03-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|72.48||2025-03-12 05:32:25.000000 UTC|2025-03-12 05:40:59.000000 UTC|86||0|26|29|0|29||0|0|
|5260|US|CDT|-18000|2025-03-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|72.48||2025-03-12 05:32:25.000000 UTC|2025-03-12 05:40:59.000000 UTC|86||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-03-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|72.48||2025-03-12 05:32:25.000000 UTC|2025-03-12 05:40:59.000000 UTC|86||0|26|0|0|18||0|0|
|5260|US|CST|-21600|2025-03-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|71.46||2025-03-07 06:30:56.000000 UTC|2025-03-07 06:39:12.000000 UTC|71||0|26|0|0|22||0|0|
|5260|US|CST|-21600|2025-03-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|71.46||2025-03-07 06:30:56.000000 UTC|2025-03-07 06:39:12.000000 UTC|71||0|26|0|0|13||0|0|
|5260|US|CST|-21600|2025-03-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|71.46||2025-03-07 06:30:56.000000 UTC|2025-03-07 06:39:12.000000 UTC|71||0|26|26|0|26||0|0|
|5260|US|CST|-21600|2025-03-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|71.46||2025-03-07 06:30:56.000000 UTC|2025-03-07 06:39:12.000000 UTC|71||0|26|25|0|25||0|0|
|5260|US|CST|-21600|2025-03-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|83.16||2025-03-08 06:30:39.000000 UTC|2025-03-08 06:39:11.000000 UTC|191||0|26|30|0|30||0|0|
|5260|US|CST|-21600|2025-03-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|83.16||2025-03-08 06:30:39.000000 UTC|2025-03-08 06:39:11.000000 UTC|191||0|26|0|0|24||0|0|
|5260|US|CST|-21600|2025-03-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|83.16||2025-03-08 06:30:39.000000 UTC|2025-03-08 06:39:11.000000 UTC|191||0|26|18|0|18||0|0|
|5260|US|CST|-21600|2025-03-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|83.16||2025-03-08 06:30:39.000000 UTC|2025-03-08 06:39:11.000000 UTC|191||0|26|30|0|30||0|0|
|5260|US|CDT|-18000|2025-03-29|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|81.55||2025-03-29 05:31:48.000000 UTC|2025-03-29 05:40:17.000000 UTC|137||0|26|25|0|25||0|0|
|5260|US|CDT|-18000|2025-03-29|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|81.55||2025-03-29 05:31:48.000000 UTC|2025-03-29 05:40:17.000000 UTC|137||0|26|33|0|33||0|0|
|5260|US|CDT|-18000|2025-03-29|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|81.55||2025-03-29 05:31:48.000000 UTC|2025-03-29 05:40:17.000000 UTC|137||0|26|25|0|25||0|0|
|5260|US|CDT|-18000|2025-03-29|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|81.55||2025-03-29 05:31:48.000000 UTC|2025-03-29 05:40:17.000000 UTC|137||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-03-30|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|96.84||2025-03-30 05:30:40.000000 UTC|2025-03-30 05:38:56.000000 UTC|139||0|26|23|0|23||0|0|
|5260|US|CDT|-18000|2025-03-30|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|96.84||2025-03-30 05:30:40.000000 UTC|2025-03-30 05:38:56.000000 UTC|139||0|26|0|0|23||0|0|
|5260|US|CDT|-18000|2025-03-30|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|96.84||2025-03-30 05:30:40.000000 UTC|2025-03-30 05:38:56.000000 UTC|139||0|26|38|0|38||0|0|
|5260|US|CDT|-18000|2025-03-30|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|96.84||2025-03-30 05:30:40.000000 UTC|2025-03-30 05:38:56.000000 UTC|139||0|26|34|0|34||0|0|
|5260|US|CDT|-18000|2025-03-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|72.48||2025-03-13 05:30:25.000000 UTC|2025-03-13 05:38:42.000000 UTC|0||0|26|20|0|20||0|0|
|5260|US|CDT|-18000|2025-03-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|72.48||2025-03-13 05:30:25.000000 UTC|2025-03-13 05:38:42.000000 UTC|0||0|26|32|0|32||0|0|
|5260|US|CDT|-18000|2025-03-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|72.48||2025-03-13 05:30:25.000000 UTC|2025-03-13 05:38:42.000000 UTC|0||0|26|20|0|20||0|0|
|5260|US|CDT|-18000|2025-03-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|72.48||2025-03-13 05:30:25.000000 UTC|2025-03-13 05:38:42.000000 UTC|0||0|26|0|0|20||0|0|
|5260|US|CDT|-18000|2025-03-25|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|82.41||2025-03-25 05:31:53.000000 UTC|2025-03-25 05:40:22.000000 UTC|111||0|26|28|0|28||0|0|
|5260|US|CDT|-18000|2025-03-25|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|82.41||2025-03-25 05:31:53.000000 UTC|2025-03-25 05:40:22.000000 UTC|111||0|26|38|0|19||0|0|
|5260|US|CDT|-18000|2025-03-25|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|82.41||2025-03-25 05:31:53.000000 UTC|2025-03-25 05:40:22.000000 UTC|111||0|26|0|0|26||0|0|
|5260|US|CDT|-18000|2025-03-25|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|82.41||2025-03-25 05:31:53.000000 UTC|2025-03-25 05:40:22.000000 UTC|111||0|26|26|0|26||0|0|
|5260|US|CDT|-18000|2025-03-26|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|81.19||2025-03-26 05:33:07.000000 UTC|2025-03-26 05:41:30.000000 UTC|138||0|26|0|0|19||0|0|
|5260|US|CDT|-18000|2025-03-26|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|81.19||2025-03-26 05:33:07.000000 UTC|2025-03-26 05:41:30.000000 UTC|138||0|26|0|0|36||0|0|
|5260|US|CDT|-18000|2025-03-26|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|81.19||2025-03-26 05:33:07.000000 UTC|2025-03-26 05:41:30.000000 UTC|138||0|26|0|0|25||0|0|
|5260|US|CDT|-18000|2025-03-26|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|81.19||2025-03-26 05:33:07.000000 UTC|2025-03-26 05:41:30.000000 UTC|138||0|26|20|0|19||0|0|
|5260|US|CDT|-18000|2025-03-22|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|89.72||2025-03-22 05:32:09.000000 UTC|2025-03-22 05:40:46.000000 UTC|187||0|26|0|0|24||0|0|
|5260|US|CDT|-18000|2025-03-22|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|89.72||2025-03-22 05:32:09.000000 UTC|2025-03-22 05:40:46.000000 UTC|187||0|26|24|0|24||0|0|
|5260|US|CDT|-18000|2025-03-22|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|89.72||2025-03-22 05:32:09.000000 UTC|2025-03-22 05:40:46.000000 UTC|187||0|26|0|0|30||0|0|
|5260|US|CDT|-18000|2025-03-22|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|89.72||2025-03-22 05:32:09.000000 UTC|2025-03-22 05:40:46.000000 UTC|187||0|26|31|0|31||0|0|
|5260|US|CDT|-18000|2025-04-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|67.6||2025-04-03 05:32:18.000000 UTC|2025-04-03 05:40:55.000000 UTC|145||0|26|17|0|17||0|0|
|5260|US|CDT|-18000|2025-04-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|67.6||2025-04-03 05:32:18.000000 UTC|2025-04-03 05:40:55.000000 UTC|145||0|26|0|0|17||0|0|
|5260|US|CDT|-18000|2025-04-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|67.6||2025-04-03 05:32:18.000000 UTC|2025-04-03 05:40:55.000000 UTC|145||0|26|35|0|35||0|0|
|5260|US|CDT|-18000|2025-04-03|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|67.6||2025-04-03 05:32:18.000000 UTC|2025-04-03 05:40:55.000000 UTC|145||0|26|17|0|17||0|0|
|5260|US|CDT|-18000|2025-03-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|69.42||2025-03-20 05:31:09.000000 UTC|2025-03-20 05:39:33.000000 UTC|51||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-03-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|69.42||2025-03-20 05:31:09.000000 UTC|2025-03-20 05:39:33.000000 UTC|51||0|26|0|0|19||0|0|
|5260|US|CDT|-18000|2025-03-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|69.42||2025-03-20 05:31:09.000000 UTC|2025-03-20 05:39:33.000000 UTC|51||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-03-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|69.42||2025-03-20 05:31:09.000000 UTC|2025-03-20 05:39:33.000000 UTC|51||0|26|30|0|30||0|0|
|5260|US|CDT|-18000|2025-03-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|68.9||2025-03-21 05:30:31.000000 UTC|2025-03-21 05:39:09.000000 UTC|69||0|26|20|0|20||0|0|
|5260|US|CDT|-18000|2025-03-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|68.9||2025-03-21 05:30:31.000000 UTC|2025-03-21 05:39:09.000000 UTC|69||0|26|0|0|15||0|0|
|5260|US|CDT|-18000|2025-03-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|68.9||2025-03-21 05:30:31.000000 UTC|2025-03-21 05:39:09.000000 UTC|69||0|26|24|0|24||0|0|
|5260|US|CDT|-18000|2025-03-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|68.9||2025-03-21 05:30:31.000000 UTC|2025-03-21 05:39:09.000000 UTC|69||0|26|25|0|25||0|0|
|5260|US|CDT|-18000|2025-03-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|86.46||2025-03-16 05:31:44.000000 UTC|2025-03-16 05:40:10.000000 UTC|33||0|26|0|0|19||0|0|
|5260|US|CDT|-18000|2025-03-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|86.46||2025-03-16 05:31:44.000000 UTC|2025-03-16 05:40:10.000000 UTC|33||0|26|42|0|42||0|0|
|5260|US|CDT|-18000|2025-03-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|86.46||2025-03-16 05:31:44.000000 UTC|2025-03-16 05:40:10.000000 UTC|33||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-03-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|86.46||2025-03-16 05:31:44.000000 UTC|2025-03-16 05:40:10.000000 UTC|33||0|26|25|0|25||0|0|
|5260|US|CDT|-18000|2025-03-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|80.91||2025-03-17 05:32:19.000000 UTC|2025-03-17 05:41:01.000000 UTC|28||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-03-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|80.91||2025-03-17 05:32:19.000000 UTC|2025-03-17 05:41:01.000000 UTC|28||0|26|33|0|33||0|0|
|5260|US|CDT|-18000|2025-03-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|80.91||2025-03-17 05:32:19.000000 UTC|2025-03-17 05:41:01.000000 UTC|28||0|26|0|0|25||0|0|
|5260|US|CDT|-18000|2025-03-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|80.91||2025-03-17 05:32:19.000000 UTC|2025-03-17 05:41:01.000000 UTC|28||0|26|19|0|19||0|0|
|5260|US|CST|-21600|2025-03-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|73.64||2025-03-04 06:30:40.000000 UTC|2025-03-04 06:39:04.000000 UTC|120||0|26|19|0|19||0|0|
|5260|US|CST|-21600|2025-03-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|73.64||2025-03-04 06:30:40.000000 UTC|2025-03-04 06:39:04.000000 UTC|120||0|26|25|0|25||0|0|
|5260|US|CST|-21600|2025-03-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|73.64||2025-03-04 06:30:40.000000 UTC|2025-03-04 06:39:04.000000 UTC|120||0|26|19|0|19||0|0|
|5260|US|CST|-21600|2025-03-04|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|73.64||2025-03-04 06:30:40.000000 UTC|2025-03-04 06:39:04.000000 UTC|120||0|26|27|0|27||0|0|
|5260|US|CST|-21600|2025-03-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|72.55||2025-03-05 06:31:50.000000 UTC|2025-03-05 06:39:56.000000 UTC|81||0|26|23|0|23||0|0|
|5260|US|CST|-21600|2025-03-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|72.55||2025-03-05 06:31:50.000000 UTC|2025-03-05 06:39:56.000000 UTC|81||0|26|24|0|23||0|0|
|5260|US|CST|-21600|2025-03-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|72.55||2025-03-05 06:31:50.000000 UTC|2025-03-05 06:39:56.000000 UTC|81||0|26|54|0|27||0|0|
|5260|US|CST|-21600|2025-03-05|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|72.55||2025-03-05 06:31:50.000000 UTC|2025-03-05 06:39:56.000000 UTC|81||0|26|0|0|17||0|0|
|5260|US|CDT|-18000|2025-04-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|63.37||2025-04-15 05:30:34.000000 UTC|2025-04-15 05:38:21.000000 UTC|189||0|26|27|0|27||0|0|
|5260|US|CDT|-18000|2025-04-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|63.37||2025-04-15 05:30:34.000000 UTC|2025-04-15 05:38:21.000000 UTC|189||0|26|17|0|17||0|0|
|5260|US|CDT|-18000|2025-04-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|63.37||2025-04-15 05:30:34.000000 UTC|2025-04-15 05:38:21.000000 UTC|189||0|26|17|0|17||0|0|
|5260|US|CDT|-18000|2025-04-15|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|63.37||2025-04-15 05:30:34.000000 UTC|2025-04-15 05:38:21.000000 UTC|189||0|26|0|0|17||0|0|
|5260|US|CDT|-18000|2025-04-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|63.37||2025-04-16 05:30:18.000000 UTC|2025-04-16 05:38:12.000000 UTC|166||0|26|29|0|29||0|0|
|5260|US|CDT|-18000|2025-04-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|63.37||2025-04-16 05:30:18.000000 UTC|2025-04-16 05:38:12.000000 UTC|166||0|26|17|0|17||0|0|
|5260|US|CDT|-18000|2025-04-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|63.37||2025-04-16 05:30:18.000000 UTC|2025-04-16 05:38:12.000000 UTC|166||0|26|0|0|17||0|0|
|5260|US|CDT|-18000|2025-04-16|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|63.37||2025-04-16 05:30:18.000000 UTC|2025-04-16 05:38:12.000000 UTC|166||0|26|0|0|17||0|0|
|5260|US|CDT|-18000|2025-04-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|63.85||2025-04-17 05:31:49.000000 UTC|2025-04-17 05:39:36.000000 UTC|176||0|26|31|0|31||0|0|
|5260|US|CDT|-18000|2025-04-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|63.85||2025-04-17 05:31:49.000000 UTC|2025-04-17 05:39:36.000000 UTC|176||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-04-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|63.85||2025-04-17 05:31:49.000000 UTC|2025-04-17 05:39:36.000000 UTC|176||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-04-17|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|63.85||2025-04-17 05:31:49.000000 UTC|2025-04-17 05:39:36.000000 UTC|176||0|26|18|0|18||0|0|
|5260|US|CDT|-18000|2025-04-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|75.46||2025-04-18 05:32:25.000000 UTC|2025-04-18 05:40:18.000000 UTC|213||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-04-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|75.46||2025-04-18 05:32:25.000000 UTC|2025-04-18 05:40:18.000000 UTC|213||0|26|32|0|32||0|0|
|5260|US|CDT|-18000|2025-04-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|75.46||2025-04-18 05:32:25.000000 UTC|2025-04-18 05:40:18.000000 UTC|213||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-04-18|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|75.46||2025-04-18 05:32:25.000000 UTC|2025-04-18 05:40:18.000000 UTC|213||0|26|0|0|21||0|0|
|5260|US|CDT|-18000|2025-04-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|69.12||2025-04-01 05:32:20.000000 UTC|2025-04-01 05:40:47.000000 UTC|69||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-04-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|69.12||2025-04-01 05:32:20.000000 UTC|2025-04-01 05:40:47.000000 UTC|69||0|26|0|0|19||0|0|
|5260|US|CDT|-18000|2025-04-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|69.12||2025-04-01 05:32:20.000000 UTC|2025-04-01 05:40:47.000000 UTC|69||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-04-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|69.12||2025-04-01 05:32:20.000000 UTC|2025-04-01 05:40:47.000000 UTC|69||0|26|28|0|28||0|0|
|5260|US|CDT|-18000|2025-04-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|68.1||2025-04-02 05:31:44.000000 UTC|2025-04-02 05:39:55.000000 UTC|158||0|26|0|0|17||0|0|
|5260|US|CDT|-18000|2025-04-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|68.1||2025-04-02 05:31:44.000000 UTC|2025-04-02 05:39:55.000000 UTC|158||0|26|17|0|17||0|0|
|5260|US|CDT|-18000|2025-04-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|68.1||2025-04-02 05:31:44.000000 UTC|2025-04-02 05:39:55.000000 UTC|158||0|26|34|0|34||0|0|
|5260|US|CDT|-18000|2025-04-02|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|68.1||2025-04-02 05:31:44.000000 UTC|2025-04-02 05:39:55.000000 UTC|158||0|26|0|0|17||0|0|
|5260|US|CST|-21600|2025-02-22|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|79.89||2025-02-22 06:32:41.000000 UTC|2025-02-22 06:41:15.000000 UTC|181||0|28|35|0|35||0|0|
|5260|US|CST|-21600|2025-02-22|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|79.89||2025-02-22 06:32:41.000000 UTC|2025-02-22 06:41:15.000000 UTC|181||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-22|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|79.89||2025-02-22 06:32:41.000000 UTC|2025-02-22 06:41:15.000000 UTC|181||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-22|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|79.89||2025-02-22 06:32:41.000000 UTC|2025-02-22 06:41:15.000000 UTC|181||0|28|0|0|22||0|0|
|5260|US|CDT|-18000|2025-04-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|71.27||2025-04-12 05:32:00.000000 UTC|2025-04-12 05:39:59.000000 UTC|131||0|26|20|0|20||0|0|
|5260|US|CDT|-18000|2025-04-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|71.27||2025-04-12 05:32:00.000000 UTC|2025-04-12 05:39:59.000000 UTC|131||0|26|34|0|34||0|0|
|5260|US|CDT|-18000|2025-04-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|71.27||2025-04-12 05:32:00.000000 UTC|2025-04-12 05:39:59.000000 UTC|131||0|26|0|0|20||0|0|
|5260|US|CDT|-18000|2025-04-12|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|71.27||2025-04-12 05:32:00.000000 UTC|2025-04-12 05:39:59.000000 UTC|131||0|26|40|0|20||0|0|
|5260|US|CDT|-18000|2025-04-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|73.15||2025-04-13 05:31:42.000000 UTC|2025-04-13 05:39:35.000000 UTC|121||0|26|32|0|32||0|0|
|5260|US|CDT|-18000|2025-04-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|73.15||2025-04-13 05:31:42.000000 UTC|2025-04-13 05:39:35.000000 UTC|121||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-04-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|73.15||2025-04-13 05:31:42.000000 UTC|2025-04-13 05:39:35.000000 UTC|121||0|26|21|0|21||0|0|
|5260|US|CDT|-18000|2025-04-13|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|73.15||2025-04-13 05:31:42.000000 UTC|2025-04-13 05:39:35.000000 UTC|121||0|26|0|0|16||0|0|
|5260|US|CDT|-18000|2025-04-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|81.76||2025-04-20 05:31:35.000000 UTC|2025-04-20 05:39:30.000000 UTC|157||0|26|0|0|34||0|0|
|5260|US|CDT|-18000|2025-04-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|81.76||2025-04-20 05:31:35.000000 UTC|2025-04-20 05:39:30.000000 UTC|157||0|26|22|0|22||0|0|
|5260|US|CDT|-18000|2025-04-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|81.76||2025-04-20 05:31:35.000000 UTC|2025-04-20 05:39:30.000000 UTC|157||0|26|22|0|22||0|0|
|5260|US|CDT|-18000|2025-04-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|81.76||2025-04-20 05:31:35.000000 UTC|2025-04-20 05:39:30.000000 UTC|157||0|26|0|0|22||0|0|
|5260|US|CDT|-18000|2025-04-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|82.78||2025-04-21 05:32:01.000000 UTC|2025-04-21 05:39:52.000000 UTC|56||0|26|23|0|23||0|0|
|5260|US|CDT|-18000|2025-04-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|82.78||2025-04-21 05:32:01.000000 UTC|2025-04-21 05:39:52.000000 UTC|56||0|26|13|0|33||0|0|
|5260|US|CDT|-18000|2025-04-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|82.78||2025-04-21 05:32:01.000000 UTC|2025-04-21 05:39:52.000000 UTC|56||0|26|23|0|23||0|0|
|5260|US|CDT|-18000|2025-04-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|82.78||2025-04-21 05:32:01.000000 UTC|2025-04-21 05:39:52.000000 UTC|56||0|26|24|0|23||0|0|
|5260|US|CST|-21600|2025-02-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|74.17||2025-02-07 06:30:21.000000 UTC|2025-02-07 06:39:51.000000 UTC|81||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|74.17||2025-02-07 06:30:21.000000 UTC|2025-02-07 06:39:51.000000 UTC|81||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|74.17||2025-02-07 06:30:21.000000 UTC|2025-02-07 06:39:51.000000 UTC|81||0|28|33|0|33||0|0|
|5260|US|CST|-21600|2025-02-07|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|74.17||2025-02-07 06:30:21.000000 UTC|2025-02-07 06:39:51.000000 UTC|81||0|28|0|0|16||0|0|
|5260|US|CST|-21600|2025-02-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|81.98||2025-02-08 06:30:31.000000 UTC|2025-02-08 06:39:20.000000 UTC|201||0|28|33|0|33||0|0|
|5260|US|CST|-21600|2025-02-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|81.98||2025-02-08 06:30:31.000000 UTC|2025-02-08 06:39:20.000000 UTC|201||0|28|0|0|19||0|0|
|5260|US|CST|-21600|2025-02-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|81.98||2025-02-08 06:30:31.000000 UTC|2025-02-08 06:39:20.000000 UTC|201||0|28|0|0|25||0|0|
|5260|US|CST|-21600|2025-02-08|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|81.98||2025-02-08 06:30:31.000000 UTC|2025-02-08 06:39:20.000000 UTC|201||0|28|25|0|25||0|0|
|5260|US|CST|-21600|2025-02-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|57.62||2025-02-20 06:30:56.000000 UTC|2025-02-20 06:40:04.000000 UTC|140||0|28|16|0|16||0|0|
|5260|US|CST|-21600|2025-02-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|57.62||2025-02-20 06:30:56.000000 UTC|2025-02-20 06:40:04.000000 UTC|140||0|28|0|0|16||0|0|
|5260|US|CST|-21600|2025-02-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|57.62||2025-02-20 06:30:56.000000 UTC|2025-02-20 06:40:04.000000 UTC|140||0|28|16|0|16||0|0|
|5260|US|CST|-21600|2025-02-20|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|57.62||2025-02-20 06:30:56.000000 UTC|2025-02-20 06:40:04.000000 UTC|140||0|28|0|0|28||0|0|
|5260|US|CST|-21600|2025-02-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|65.22||2025-02-21 06:31:05.000000 UTC|2025-02-21 06:40:29.000000 UTC|79||0|28|0|0|17||0|0|
|5260|US|CST|-21600|2025-02-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|65.22||2025-02-21 06:31:05.000000 UTC|2025-02-21 06:40:29.000000 UTC|79||0|28|27|0|27||0|0|
|5260|US|CST|-21600|2025-02-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|65.22||2025-02-21 06:31:05.000000 UTC|2025-02-21 06:40:29.000000 UTC|79||0|28|14|0|17||0|0|
|5260|US|CST|-21600|2025-02-21|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|65.22||2025-02-21 06:31:05.000000 UTC|2025-02-21 06:40:29.000000 UTC|79||0|28|22|0|22||0|0|
|5260|US|CST|-21600|2025-02-27|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|67.29||2025-02-27 06:30:54.000000 UTC|2025-02-27 06:39:00.000000 UTC|182||0|28|28|0|28||0|0|
|5260|US|CST|-21600|2025-02-27|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|67.29||2025-02-27 06:30:54.000000 UTC|2025-02-27 06:39:00.000000 UTC|182||0|28|0|0|17||0|0|
|5260|US|CST|-21600|2025-02-27|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|67.29||2025-02-27 06:30:54.000000 UTC|2025-02-27 06:39:00.000000 UTC|182||0|28|23|0|23||0|0|
|5260|US|CST|-21600|2025-02-27|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|67.29||2025-02-27 06:30:54.000000 UTC|2025-02-27 06:39:00.000000 UTC|182||0|28|17|0|17||0|0|
|5260|US|CST|-21600|2025-02-28|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|68.74||2025-02-28 06:30:53.000000 UTC|2025-02-28 06:39:17.000000 UTC|100||0|26|16|0|16||0|0|
|5260|US|CST|-21600|2025-02-28|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|68.74||2025-02-28 06:30:53.000000 UTC|2025-02-28 06:39:17.000000 UTC|100||0|26|25|0|25||0|0|
|5260|US|CST|-21600|2025-02-28|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|68.74||2025-02-28 06:30:53.000000 UTC|2025-02-28 06:39:17.000000 UTC|100||0|26|54|0|27||0|0|
|5260|US|CST|-21600|2025-02-28|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|68.74||2025-02-28 06:30:53.000000 UTC|2025-02-28 06:39:17.000000 UTC|100||0|26|0|0|16||0|0|
|5260|US|CDT|-18000|2025-04-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|59.03||2025-04-09 05:30:41.000000 UTC|2025-04-09 05:38:54.000000 UTC|32||0|26|20|0|20||0|0|
|5260|US|CDT|-18000|2025-04-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|59.03||2025-04-09 05:30:41.000000 UTC|2025-04-09 05:38:54.000000 UTC|32||0|26|15|0|15||0|0|
|5260|US|CDT|-18000|2025-04-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|59.03||2025-04-09 05:30:41.000000 UTC|2025-04-09 05:38:54.000000 UTC|32||0|26|0|0|15||0|0|
|5260|US|CDT|-18000|2025-04-09|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|59.03||2025-04-09 05:30:41.000000 UTC|2025-04-09 05:38:54.000000 UTC|32||0|26|28|0|28||0|0|
|5260|US|CDT|-18000|2025-04-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|59.03||2025-04-10 05:32:15.000000 UTC|2025-04-10 05:39:54.000000 UTC|11||0|26|16|0|16||0|0|
|5260|US|CDT|-18000|2025-04-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|59.03||2025-04-10 05:32:15.000000 UTC|2025-04-10 05:39:54.000000 UTC|11||0|26|16|0|16||0|0|
|5260|US|CDT|-18000|2025-04-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|59.03||2025-04-10 05:32:15.000000 UTC|2025-04-10 05:39:54.000000 UTC|11||0|26|16|0|16||0|0|
|5260|US|CDT|-18000|2025-04-10|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|59.03||2025-04-10 05:32:15.000000 UTC|2025-04-10 05:39:54.000000 UTC|11||0|26|30|0|30||0|0|
|5260|US|CDT|-18000|2025-04-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|68.69||2025-04-14 05:30:49.000000 UTC|2025-04-14 05:39:12.000000 UTC|84||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-04-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|68.69||2025-04-14 05:30:49.000000 UTC|2025-04-14 05:39:12.000000 UTC|84||0|26|19|0|19||0|0|
|5260|US|CDT|-18000|2025-04-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|68.69||2025-04-14 05:30:49.000000 UTC|2025-04-14 05:39:12.000000 UTC|84||0|26|29|0|29||0|0|
|5260|US|CDT|-18000|2025-04-14|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|68.69||2025-04-14 05:30:49.000000 UTC|2025-04-14 05:39:12.000000 UTC|84||0|26|0|0|19||0|0|
|5260|US|CST|-21600|2025-03-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|203|15:00:00|17:59:59|79.31||2025-03-01 06:32:36.000000 UTC|2025-03-01 06:40:54.000000 UTC|81||0|26|17|0|17||0|0|
|5260|US|CST|-21600|2025-03-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|202|12:00:00|14:59:59|79.31||2025-03-01 06:32:36.000000 UTC|2025-03-01 06:40:54.000000 UTC|81||0|26|27|0|23||0|0|
|5260|US|CST|-21600|2025-03-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|204|18:00:00|20:59:59|79.31||2025-03-01 06:32:36.000000 UTC|2025-03-01 06:40:54.000000 UTC|81||0|26|0|0|23||0|0|
|5260|US|CST|-21600|2025-03-01|7874237408|FG TRAD 36OZ|1|Made-To-Stock|3|Rotisserie|04:00:00|02:00:00|201|09:00:00|11:59:59|79.31||2025-03-01 06:32:36.000000 UTC|2025-03-01 06:40:54.000000 UTC|81||0|26|36|0|36||0|0|

---

#### Dataset 2: Markdown
*This dataset captures the number of throwaway events. The throw away event (column: event_type) indicates waste. The column: MUMD_QTY shows number of items that were throw away/wasted .*

|REGION_NBR|MARKET_NBR|STORE_NBR|ACCTG_DEPT_NBR|ACCTG_DEPT_DESC|DEPT_CATG_GRP_NBR|DEPT_CATG_GRP_DESC|ITEM_NBR|ITEM_DESCRIPTION|action_date|event_type|weight|CURR_RTL_AMT|PREV_RTL_AMT|MUMD_QTY|MUMD_AMT|LY_454_MUMD_AMT|LY_454_PREV_RTL_AMT|LY_CAL_CURR_RTL_AMT|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-19|throwaway|42.525|0|125.37|21|125.37|17.91|17.91|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-10|throwaway|20.25|0|59.7|10|59.7|23.88|23.88|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-02|throwaway|0.0|0|0|0|0|41.79|41.79|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-13|throwaway|32.4|0|95.52|16|95.52|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-03-29|throwaway|0.0|0|0|0|0|5.97|5.97|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-05|throwaway|2.025|0|5.97|1|5.97|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-20|throwaway|42.525|0|125.37|21|125.37|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-22|throwaway|0.0|0|0|0|0|35.82|35.82|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-09|throwaway|52.65|0|155.22|26|155.22|35.82|35.82|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-03-26|throwaway|30.375|0|89.55|15|89.55|107.46|107.46|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-03-31|throwaway|18.225|0|53.73|9|53.73|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-07|throwaway|0.0|0|0|0|0|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-17|throwaway|89.1|0|262.68|44|262.68|5.97|5.97|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-03-27|throwaway|2.025|0|5.97|1|5.97|47.76|47.76|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-11|throwaway|8.1|0|23.88|4|23.88|11.94|11.94|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-03-25|throwaway|30.375|0|89.55|15|89.55|65.67|65.67|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-16|throwaway|10.125|0|29.85|5|29.85|23.88|23.88|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-21|throwaway|26.325|0|77.61|13|77.61|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-14|throwaway|2.025|0|5.97|1|5.97|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-12|throwaway|0.0|0|0|0|0|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-06|throwaway|20.25|0|59.7|10|59.7|5.97|5.97|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-03-28|throwaway|2.025|0|5.97|1|5.97|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-08|throwaway|38.475|0|113.43|19|113.43|17.91|17.91|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-03|throwaway|8.1|0|23.88|4|23.88|5.97|5.97|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-01|throwaway|28.35|0|83.58|14|83.58|11.94|11.94|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-04|throwaway|14.175|0|41.79|7|41.79|0|0|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-18|throwaway|6.075|0|17.91|3|17.91|53.73|53.73|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-04-15|throwaway|12.15|0|35.82|6|35.82|65.67|65.67|0|
|42|323|5260|80|SERVICE DELI|6602|ROTISSERIE|595342628|FG TRAD 36OZ|2025-03-30|throwaway|2.025|0|5.97|1|5.97|5.97|5.97|0|
---

#### Dataset 3: Sales History
*This dataset represents the number of rotisserie chickens that were sold on a paritcular date The column SALES_UNIT_QTY has the quantity of rotisserie chickens sold. The column visit_dt indicates the date it was sold on.*

|STORE_NBR|MKT_NBR|ITEM_NBR|ITEM_DESC_1|VEND_NBR|VEND_NM|DEPT_CATG_DESC|DEPT_CATG_GRP_DESC|DEPT_CATG_NBR|ACCTG_DEPT_NBR|ACCTG_DEPT_DESC|WHSE_ALGN_TYPE_CD|UPSTRM_DC_NBR|PRMRY_DC_NBR|VISIT_DT|LY_COMP_VISIT_DT|LY_454_SCAN_CNT|SALES_AMT|LY_CAL_SALES_AMT|LY_454_SALES_AMT|SALES_UNIT_QTY|LY_CAL_SALES_UNIT_QTY|LY_454_SALES_UNIT_QTY|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-06|2024-02-08|17|59.7|83.58|101.49|10|14|17|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-04|2024-02-06|33|256.71|244.77|197.01|43|41|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-08|2024-03-09|4|35.82|17.91|23.88|6|3|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-09|2024-04-10|0|41.79|0|0|7|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-24|2024-03-25|1|5.97|0|11.94|1|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-13|2024-03-14|0|71.64|0|0|12|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-26|2024-03-27|8|35.82|59.7|47.76|6|10|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-17|2024-04-18|9|11.94|35.82|65.67|2|6|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-31|2024-02-02|0|35.82|5.97|0|6|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-10|2024-02-12|6|95.52|95.52|35.82|16|16|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-12|2024-03-13|4|11.94|5.97|23.88|2|1|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-27|2024-02-29|8|53.73|47.76|65.67|9|8|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-18|2024-03-19|2|11.94|11.94|11.94|2|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-24|2024-03-25|11|23.88|89.55|65.67|4|15|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-07|2024-03-08|5|29.85|47.76|29.85|5|8|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-09|2024-04-10|10|53.73|83.58|65.67|9|14|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-08|2024-04-09|13|101.49|143.28|161.19|17|24|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-22|2024-01-24|1|59.7|0|23.88|10|0|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-05|2024-04-06|3|11.94|0|29.85|2|0|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-27|2024-01-29|1|23.88|41.79|5.97|4|7|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-27|2024-02-29|9|35.82|71.64|53.73|6|12|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-27|2024-03-28|9|35.82|71.64|53.73|6|12|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-10|2024-02-12|1|47.76|17.91|5.97|8|3|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-22|2024-02-24|0|155.22|0|0|26|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-19|2024-04-20|24|202.98|47.76|143.28|34|8|24|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-04|2024-04-05|4|17.91|5.97|29.85|3|1|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-12|2024-03-13|1|0|5.97|5.97|0|1|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-09|2024-03-10|17|155.22|119.4|101.49|26|20|17|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-14|2024-04-15|4|23.88|17.91|35.82|4|3|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-15|2024-04-16|29|226.86|214.92|173.13|38|36|29|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-03|2024-04-04|2|17.91|17.91|23.88|3|3|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-20|2024-04-21|43|262.68|202.98|256.71|44|34|43|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-05|2024-02-07|10|23.88|95.52|59.7|4|16|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-14|2024-02-16|39|161.19|119.4|232.83|27|20|39|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-20|2024-02-22|29|119.4|197.01|173.13|20|33|29|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-03|2024-02-05|5|65.67|29.85|29.85|11|5|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-15|2024-02-17|39|179.1|143.28|232.83|30|24|39|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-06|2024-03-07|31|226.86|161.19|185.07|38|27|31|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-22|2024-03-23|0|17.91|0|0|3|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-07|2024-02-09|5|95.52|65.67|113.43|16|11|19|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-25|2024-01-27|4|29.85|11.94|23.88|5|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-19|2024-02-21|12|0|101.49|71.64|0|17|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-03|2024-04-04|36|268.65|232.83|214.92|45|39|36|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-19|2024-03-20|9|47.76|47.76|53.73|8|8|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-25|2024-03-26|9|47.76|125.37|107.46|8|21|18|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-05|2024-03-06|0|23.88|0|0|4|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-08|2024-02-10|16|125.37|101.49|95.52|21|17|16|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-01|2024-03-02|2|41.79|35.82|23.88|7|6|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-09|2024-02-11|41|161.19|161.19|244.77|27|27|41|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-15|2024-04-16|8|23.88|59.7|59.7|4|10|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-13|2024-04-14|34|197.01|161.19|202.98|33|27|34|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-11|2024-04-12|5|23.88|35.82|29.85|4|6|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-05|2024-03-06|27|197.01|197.01|161.19|33|33|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-22|2024-01-24|10|59.7|41.79|59.7|10|7|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-13|2024-02-15|2|17.91|29.85|11.94|3|5|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-22|2024-03-23|10|17.91|95.52|89.55|3|16|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-23|2024-01-25|0|59.7|29.85|0|10|5|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-27|2024-01-29|7|47.76|101.49|41.79|8|17|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-14|2024-04-15|13|71.64|149.25|77.61|12|25|13|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-26|2024-02-28|36|179.1|191.04|214.92|30|32|36|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-16|2024-04-17|35|232.83|173.13|208.95|39|29|35|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-30|2024-03-31|3|17.91|11.94|17.91|3|2|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-01|2024-04-02|1|5.97|35.82|11.94|1|6|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-21|2024-03-22|2|35.82|41.79|23.88|6|7|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-23|2024-03-24|24|107.46|131.34|143.28|18|22|24|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-01|2024-03-02|4|101.49|35.82|53.73|17|6|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-20|2024-02-22|5|11.94|47.76|29.85|2|8|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-26|2024-01-28|7|17.91|47.76|47.76|3|8|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-07|2024-04-08|5|23.88|65.67|35.82|4|11|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-24|2024-01-26|7|65.67|53.73|89.55|11|9|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-14|2024-03-15|1|17.91|0|5.97|3|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-08|2024-02-10|12|29.85|17.91|83.58|5|3|14|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-14|2024-03-15|8|29.85|35.82|53.73|5|6|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-26|2024-02-28|0|23.88|0|0|4|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-05|2024-02-07|4|23.88|29.85|23.88|4|5|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-06|2024-02-08|0|47.76|23.88|0|8|4|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-07|2024-04-08|2|29.85|47.76|11.94|5|8|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-23|2024-03-24|3|17.91|23.88|17.91|3|4|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-06|2024-02-08|3|11.94|17.91|17.91|2|3|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-31|2024-04-01|0|41.79|0|0|7|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-17|2024-04-18|4|11.94|11.94|23.88|2|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-21|2024-03-22|31|95.52|161.19|185.07|16|27|31|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-24|2024-02-26|12|11.94|113.43|71.64|2|19|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-28|2024-03-29|1|35.82|11.94|11.94|6|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-10|2024-03-11|7|23.88|29.85|47.76|4|5|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-13|2024-02-15|5|17.91|11.94|41.79|3|2|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-13|2024-02-15|10|41.79|71.64|59.7|7|12|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-27|2024-03-28|2|5.97|11.94|11.94|1|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-20|2024-02-22|11|23.88|53.73|65.67|4|9|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-31|2024-04-01|2|47.76|11.94|11.94|8|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-22|2024-01-24|9|17.91|17.91|53.73|3|3|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-25|2024-03-26|0|41.79|0|0|7|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-23|2024-02-25|11|89.55|53.73|65.67|15|9|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-28|2024-03-01|1|23.88|29.85|11.94|4|5|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-07|2024-04-08|1|0|0|5.97|0|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-03|2024-04-04|0|17.91|0|0|3|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-05|2024-04-06|27|232.83|232.83|161.19|39|39|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-09|2024-03-10|7|113.43|53.73|41.79|19|9|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-03|2024-02-05|40|232.83|197.01|238.8|39|33|40|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-26|2024-03-27|1|5.97|11.94|11.94|1|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-25|2024-03-26|1|23.88|11.94|17.91|4|2|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-26|2024-02-28|1|23.88|29.85|17.91|4|5|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-29|2024-01-31|2|11.94|41.79|11.94|2|7|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-26|2024-02-28|11|29.85|71.64|65.67|5|12|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-07|2024-04-08|30|244.77|208.95|179.1|41|35|30|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-19|2024-02-21|35|0|173.13|208.95|0|29|35|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-10|2024-03-11|9|59.7|101.49|53.73|10|17|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-18|2024-04-19|31|167.16|185.07|185.07|28|31|31|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-30|2024-02-01|8|41.79|23.88|53.73|7|4|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-21|2024-03-22|14|59.7|71.64|83.58|10|12|14|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-09|2024-03-10|5|41.79|23.88|29.85|7|4|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-23|2024-03-24|0|71.64|23.88|0|12|4|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-10|2024-04-11|5|23.88|65.67|35.82|4|11|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-17|2024-02-19|29|286.56|232.83|173.13|48|39|29|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-17|2024-03-18|15|29.85|53.73|89.55|5|9|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-11|2024-02-13|28|220.89|244.77|167.16|37|41|28|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-01|2024-04-02|4|35.82|11.94|23.88|6|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-23|2024-01-25|5|23.88|59.7|65.67|4|10|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-24|2024-02-26|32|173.13|173.13|191.04|29|29|32|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-09|2024-03-10|31|274.62|179.1|185.07|46|30|31|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-20|2024-03-21|27|250.74|208.95|161.19|42|35|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-28|2024-03-29|11|41.79|59.7|71.64|7|10|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-12|2024-04-13|4|23.88|11.94|23.88|4|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-01|2024-02-03|3|23.88|11.94|17.91|4|2|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-03|2024-04-04|7|23.88|29.85|41.79|4|5|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-08|2024-04-09|1|11.94|35.82|5.97|2|6|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-28|2024-03-01|23|173.13|214.92|137.31|29|36|23|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-27|2024-03-28|0|41.79|5.97|0|7|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-25|2024-03-26|4|29.85|53.73|29.85|5|9|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-21|2024-04-22|21|29.85|113.43|125.37|5|19|21|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-17|2024-02-19|6|59.7|11.94|35.82|10|2|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-19|2024-04-20|3|23.88|35.82|41.79|4|6|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-20|2024-04-21|5|23.88|23.88|35.82|4|4|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-15|2024-03-16|4|71.64|65.67|59.7|12|11|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-30|2024-02-01|6|59.7|77.61|35.82|10|13|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-02|2024-04-03|2|5.97|11.94|17.91|1|2|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-19|2024-04-20|34|268.65|185.07|202.98|45|31|34|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-03|2024-04-04|1|41.79|35.82|5.97|7|6|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-30|2024-03-31|2|107.46|23.88|23.88|18|4|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-09|2024-02-11|8|83.58|47.76|47.76|14|8|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-14|2024-03-15|13|41.79|71.64|77.61|7|12|13|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-02|2024-02-04|41|202.98|137.31|244.77|34|23|41|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-15|2024-02-17|0|89.55|5.97|0|15|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-28|2024-01-30|4|17.91|65.67|23.88|3|11|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-29|2024-01-31|1|5.97|0|5.97|1|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-25|2024-01-27|17|101.49|59.7|101.49|17|10|17|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-07|2024-04-08|12|71.64|107.46|71.64|12|18|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-14|2024-04-15|36|268.65|202.98|214.92|45|34|36|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-01|2024-02-03|5|59.7|83.58|59.7|10|14|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-31|2024-02-02|15|53.73|53.73|89.55|9|9|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-08|2024-03-09|7|23.88|29.85|53.73|4|5|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-18|2024-02-20|33|250.74|197.01|197.01|42|33|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-13|2024-04-14|0|83.58|23.88|0|14|4|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-12|2024-03-13|17|250.74|161.19|101.49|42|27|17|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-28|2024-01-30|13|11.94|83.58|77.61|2|14|13|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-16|2024-02-18|0|83.58|0|0|14|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-18|2024-03-19|0|83.58|0|0|14|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-26|2024-02-28|3|5.97|17.91|17.91|1|3|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-23|2024-02-25|28|197.01|119.4|167.16|33|20|28|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-31|2024-04-01|9|47.76|29.85|53.73|8|5|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-05|2024-04-06|8|35.82|47.76|47.76|6|8|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-24|2024-03-25|0|41.79|0|0|7|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-12|2024-04-13|5|53.73|29.85|29.85|9|5|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-18|2024-03-19|8|5.97|89.55|47.76|1|15|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-26|2024-03-27|29|197.01|191.04|173.13|33|32|29|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-09|2024-02-11|1|23.88|11.94|17.91|4|2|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-02|2024-02-04|8|53.73|53.73|59.7|9|9|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-19|2024-04-20|0|53.73|0|0|9|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-02|2024-03-03|5|125.37|89.55|59.7|21|15|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-21|2024-02-23|2|17.91|23.88|11.94|3|4|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-04|2024-04-05|2|17.91|11.94|11.94|3|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-05|2024-02-07|34|208.95|238.8|202.98|35|40|34|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-16|2024-04-17|5|47.76|59.7|35.82|8|10|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-09|2024-04-10|7|5.97|29.85|47.76|1|5|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-28|2024-03-01|0|53.73|0|0|9|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-17|2024-02-19|0|107.46|5.97|0|18|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-26|2024-01-28|10|47.76|41.79|65.67|8|7|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-16|2024-02-18|1|41.79|5.97|11.94|7|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-15|2024-02-17|4|41.79|11.94|23.88|7|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-05|2024-04-06|2|23.88|29.85|11.94|4|5|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-10|2024-02-12|2|0|11.94|11.94|0|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-28|2024-03-01|8|29.85|47.76|47.76|5|8|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-25|2024-01-27|1|0|0|5.97|0|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-24|2024-02-26|0|119.4|0|0|20|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-22|2024-03-23|4|0|11.94|23.88|0|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-18|2024-03-19|6|47.76|59.7|35.82|8|10|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-12|2024-04-13|0|77.61|0|0|13|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-15|2024-02-17|1|0|0|5.97|0|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-18|2024-04-19|8|107.46|89.55|47.76|18|15|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-23|2024-02-25|6|59.7|11.94|47.76|10|2|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-06|2024-04-07|8|41.79|11.94|47.76|7|2|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-13|2024-03-14|1|29.85|29.85|11.94|5|5|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-26|2024-03-27|12|17.91|53.73|71.64|3|9|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-08|2024-03-09|30|388.05|131.34|179.1|65|22|30|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-30|2024-03-31|1|11.94|29.85|5.97|2|5|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-13|2024-03-14|4|17.91|17.91|23.88|3|3|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-14|2024-02-16|0|71.64|0|0|12|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-25|2024-01-27|0|59.7|0|0|10|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-06|2024-02-08|2|17.91|41.79|11.94|3|7|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-11|2024-04-12|9|65.67|83.58|53.73|11|14|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-08|2024-02-10|2|0|0|11.94|0|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-13|2024-03-14|25|214.92|101.49|149.25|36|17|25|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-02|2024-04-03|39|179.1|137.31|232.83|30|23|39|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-25|2024-01-27|10|47.76|29.85|71.64|8|5|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-04|2024-03-05|0|41.79|5.97|0|7|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-27|2024-03-28|41|197.01|173.13|244.77|33|29|41|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-30|2024-02-01|0|59.7|0|0|10|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-10|2024-04-11|0|47.76|5.97|0|8|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-27|2024-01-29|11|65.67|71.64|71.64|11|12|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-31|2024-02-02|3|0|11.94|17.91|0|2|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-09|2024-04-10|1|5.97|5.97|11.94|1|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-18|2024-02-20|2|53.73|83.58|35.82|9|14|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-13|2024-04-14|25|119.4|95.52|149.25|20|16|25|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-28|2024-01-30|27|185.07|208.95|161.19|31|35|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-29|2024-03-30|1|41.79|11.94|17.91|7|2|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-12|2024-03-13|11|77.61|47.76|65.67|13|8|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-01|2024-03-02|24|197.01|137.31|143.28|33|23|24|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-28|2024-03-29|0|59.7|11.94|0|10|2|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-08|2024-03-09|1|0|11.94|11.94|0|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-12|2024-02-14|0|41.79|11.94|0|7|2|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-08|2024-02-10|0|53.73|5.97|0|9|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-04|2024-02-06|0|29.85|0|0|5|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-30|2024-02-01|38|137.31|161.19|226.86|23|27|38|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-16|2024-03-17|1|29.85|11.94|5.97|5|2|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-03|2024-04-04|5|53.73|71.64|29.85|9|12|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-08|2024-02-10|3|17.91|17.91|41.79|3|3|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-17|2024-03-18|0|41.79|0|0|7|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-10|2024-02-12|0|17.91|23.88|0|3|4|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-21|2024-02-23|20|161.19|208.95|119.4|27|35|20|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-01|2024-04-02|23|173.13|179.1|137.31|29|30|23|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-03|2024-03-04|5|47.76|29.85|29.85|8|5|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-17|2024-03-18|37|268.65|155.22|220.89|45|26|37|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-27|2024-02-29|19|197.01|214.92|113.43|33|36|19|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-01|2024-04-02|2|23.88|5.97|11.94|4|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-15|2024-03-16|2|11.94|0|11.94|2|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-22|2024-01-24|0|59.7|23.88|0|10|4|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-19|2024-02-21|1|0|17.91|5.97|0|3|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-12|2024-02-14|2|41.79|23.88|23.88|7|4|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-13|2024-04-14|3|53.73|41.79|35.82|9|7|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-06|2024-02-08|42|191.04|197.01|247.81|32|33|42|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-08|2024-04-09|5|5.97|11.94|29.85|1|2|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-26|2024-03-27|0|53.73|35.82|0|9|6|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-04|2024-03-05|4|17.91|0|23.88|3|0|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-25|2024-02-27|8|29.85|59.7|47.76|5|10|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-02|2024-04-03|10|53.73|53.73|71.64|9|9|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-16|2024-03-17|4|29.85|11.94|23.88|5|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-18|2024-04-19|3|23.88|23.88|23.88|4|4|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-12|2024-02-14|20|214.92|220.89|119.4|36|37|20|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-17|2024-02-19|7|41.79|23.88|41.79|7|4|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-10|2024-04-11|14|47.76|47.76|83.58|8|8|14|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-30|2024-03-31|0|83.58|0|0|14|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-21|2024-04-22|13|47.76|59.7|83.58|8|10|14|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-20|2024-03-21|0|47.76|35.82|0|8|6|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-15|2024-02-17|12|101.49|59.7|71.64|17|10|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-05|2024-03-06|4|0|29.85|23.88|0|5|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-18|2024-04-19|1|5.97|0|5.97|1|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-11|2024-03-12|1|23.88|5.97|17.91|4|1|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-13|2024-02-15|1|11.94|0|5.97|2|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-15|2024-04-16|0|59.7|17.91|0|10|3|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-25|2024-03-26|32|202.98|202.98|191.04|34|34|32|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-03|2024-02-05|4|65.67|47.76|65.67|11|8|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-17|2024-04-18|31|173.13|208.95|185.07|29|35|31|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-22|2024-03-23|22|71.64|83.58|131.34|12|14|22|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-11|2024-02-13|2|35.82|47.76|11.94|6|8|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-21|2024-03-22|0|65.67|0|0|11|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-19|2024-04-20|1|35.82|17.91|11.94|6|3|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-30|2024-03-31|45|214.92|250.74|268.65|36|42|45|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-02|2024-02-04|6|65.67|17.91|35.82|11|3|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-11|2024-04-12|27|155.22|208.95|161.19|26|35|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-11|2024-04-12|2|17.91|5.97|11.94|3|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-13|2024-02-15|4|5.97|17.91|29.85|1|3|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-16|2024-02-18|33|202.98|232.83|197.01|34|39|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-24|2024-03-25|34|191.04|244.77|202.98|32|41|34|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-04|2024-02-06|14|29.85|89.55|83.58|5|15|14|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-22|2024-02-24|10|47.76|29.85|59.7|8|5|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-17|2024-02-19|17|191.04|71.64|101.49|32|12|17|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-04|2024-03-05|33|167.16|173.13|197.01|28|29|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-15|2024-03-16|9|89.55|77.61|53.73|15|13|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-10|2024-02-12|37|232.83|250.74|220.89|39|42|37|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-01|2024-04-02|9|35.82|53.73|53.73|6|9|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-20|2024-03-21|1|0|0|5.97|0|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-23|2024-01-25|33|161.19|155.22|197.01|27|26|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-12|2024-03-13|2|5.97|23.88|17.91|1|4|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-12|2024-03-13|3|29.85|47.76|17.91|5|8|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-08|2024-02-10|5|35.82|0|29.85|6|0|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-09|2024-02-11|6|41.79|29.85|35.82|7|5|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-29|2024-03-30|42|167.16|167.16|250.74|28|28|42|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-09|2024-02-11|7|113.43|113.43|41.79|19|19|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-22|2024-03-23|2|11.94|0|11.94|2|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-04|2024-04-05|39|83.58|214.92|232.83|14|36|39|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-17|2024-03-18|2|0|5.97|11.94|0|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-01|2024-02-03|2|17.91|0|11.94|3|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-01|2024-04-02|13|29.85|83.58|77.61|5|14|13|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-20|2024-02-22|4|11.94|11.94|23.88|2|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-06|2024-04-07|18|65.67|59.7|107.46|11|10|18|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-11|2024-02-13|4|41.79|23.88|29.85|7|4|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-09|2024-03-10|0|41.79|5.97|0|7|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-02|2024-03-03|37|191.04|143.28|220.89|32|24|37|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-25|2024-02-27|1|23.88|47.76|11.94|4|8|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-07|2024-03-08|0|95.52|0|0|16|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-06|2024-04-07|35|268.65|161.19|208.95|45|27|35|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-03|2024-02-05|0|17.91|17.91|0|3|3|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-24|2024-03-25|13|41.79|143.28|77.61|7|24|13|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-21|2024-04-22|6|23.88|35.82|41.79|4|6|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-30|2024-02-01|2|11.94|35.82|11.94|2|6|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-31|2024-02-02|2|29.85|0|17.91|5|0|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-17|2024-04-18|2|11.94|0|11.94|2|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-12|2024-02-14|11|17.91|35.82|65.67|3|6|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-15|2024-04-16|12|35.82|77.61|71.64|6|13|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-14|2024-03-15|29|179.1|149.25|173.13|30|25|29|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-09|2024-03-10|1|29.85|0|5.97|5|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-14|2024-02-16|8|47.76|65.67|47.76|8|11|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-10|2024-04-11|3|0|5.97|17.91|0|1|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-10|2024-03-11|11|53.73|41.79|65.67|9|7|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-11|2024-03-12|7|11.94|65.67|47.76|2|11|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-12|2024-04-13|27|179.1|161.19|161.19|30|27|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-21|2024-04-22|0|59.7|17.91|0|10|3|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-21|2024-03-22|1|0|0|11.94|0|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-11|2024-04-12|1|47.76|17.91|5.97|8|3|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-16|2024-03-17|26|226.86|232.83|155.22|38|39|26|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-10|2024-04-11|1|11.94|47.76|5.97|2|8|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-04|2024-02-06|1|17.91|71.64|23.88|3|12|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-09|2024-04-10|36|232.83|119.4|214.92|39|20|36|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-17|2024-03-18|10|71.64|113.43|59.7|12|19|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-14|2024-02-16|5|17.91|41.79|29.85|3|7|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-16|2024-04-17|2|11.94|53.73|29.85|2|9|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-26|2024-01-28|14|101.49|53.73|83.58|17|9|14|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-24|2024-03-25|8|35.82|41.79|101.49|6|7|17|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-01|2024-03-02|0|71.64|5.97|0|12|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-11|2024-04-12|0|41.79|0|0|7|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-11|2024-02-13|0|47.76|17.91|0|8|3|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-27|2024-01-29|46|262.68|220.89|274.62|44|37|46|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-12|2024-04-13|3|35.82|5.97|41.79|6|1|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-31|2024-02-02|5|53.73|17.91|53.73|9|3|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-20|2024-04-21|19|77.61|143.28|113.43|13|24|19|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-30|2024-02-01|1|0|0|5.97|0|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-08|2024-04-09|20|214.92|179.1|119.4|36|30|20|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-30|2024-02-01|5|5.97|29.85|29.85|1|5|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-23|2024-03-24|41|185.07|179.1|244.77|31|30|41|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-20|2024-03-21|5|17.91|11.94|29.85|3|2|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-01|2024-02-03|19|107.46|35.82|113.43|18|6|19|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-07|2024-03-08|22|167.16|185.07|131.34|28|31|22|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-28|2024-01-30|6|17.91|47.76|35.82|3|8|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-16|2024-04-17|18|5.97|71.64|107.46|1|12|18|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-03|2024-02-05|2|0|0|11.94|0|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-18|2024-02-20|0|29.85|5.97|0|5|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-29|2024-01-31|0|53.73|5.97|0|9|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-10|2024-03-11|33|208.95|185.07|197.01|35|31|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-14|2024-04-15|0|41.79|0|0|7|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-17|2024-03-18|6|11.94|41.79|35.82|2|7|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-23|2024-02-25|3|35.82|5.97|17.91|6|1|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-29|2024-01-31|3|29.85|71.64|17.91|5|12|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-01|2024-03-02|10|59.7|47.76|77.61|10|8|13|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-28|2024-03-29|28|143.28|244.77|167.16|24|41|28|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-08|2024-02-10|42|274.62|247.81|250.74|46|42|42|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-16|2024-04-17|0|11.94|0|0|2|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-07|2024-02-09|0|17.91|0|0|3|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-05|2024-04-06|0|65.67|0|0|11|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-24|2024-02-26|3|23.88|5.97|17.91|4|1|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-02|2024-03-03|1|0|0|5.97|0|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-15|2024-03-16|39|191.04|173.13|232.83|32|29|39|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-29|2024-03-30|0|29.85|0|0|5|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-02|2024-02-04|0|17.91|0|0|3|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-28|2024-01-30|0|29.85|17.91|0|5|3|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-20|2024-02-22|0|71.64|23.88|0|12|4|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-01|2024-04-02|0|29.85|5.97|0|5|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-12|2024-02-14|6|17.91|29.85|41.79|3|5|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-31|2024-04-01|1|0|23.88|11.94|0|4|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-07|2024-03-08|1|17.91|0|11.94|3|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-17|2024-02-19|8|89.55|41.79|47.76|15|7|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-18|2024-03-19|4|5.97|35.82|23.88|1|6|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-15|2024-03-16|1|5.97|5.97|11.94|1|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-07|2024-02-09|27|161.19|202.98|161.19|27|34|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-21|2024-04-22|2|23.88|47.76|23.88|4|8|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-19|2024-03-20|2|23.88|35.82|11.94|4|6|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-21|2024-02-23|9|47.76|89.55|107.46|8|15|18|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-23|2024-03-24|4|23.88|41.79|23.88|4|7|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-10|2024-03-11|1|5.97|29.85|11.94|1|5|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-19|2024-04-20|4|53.73|41.79|23.88|9|7|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-22|2024-03-23|1|5.97|11.94|11.94|1|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-25|2024-01-27|37|214.92|197.01|220.89|36|33|37|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-02|2024-04-03|0|77.61|11.94|0|13|2|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-05|2024-03-06|3|11.94|23.88|17.91|2|4|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-07|2024-04-08|10|41.79|53.73|71.64|7|9|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-12|2024-03-13|0|47.76|5.97|0|8|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-19|2024-03-20|35|185.07|274.62|208.95|31|46|35|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-28|2024-03-01|5|17.91|5.97|29.85|3|1|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-18|2024-04-19|6|47.76|65.67|41.79|8|11|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-05|2024-02-07|6|53.73|29.85|41.79|9|5|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-29|2024-01-31|9|29.85|41.79|53.73|5|7|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-02|2024-03-03|19|101.49|23.88|113.43|17|4|19|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-08|2024-03-09|0|89.55|0|0|15|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-10|2024-04-11|35|202.98|214.92|208.95|34|36|35|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-25|2024-03-26|7|35.82|65.67|59.7|6|11|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-10|2024-03-11|0|65.67|5.97|0|11|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-11|2024-03-12|27|226.86|197.01|161.19|38|33|27|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-02|2024-03-03|4|35.82|11.94|23.88|6|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-06|2024-04-07|9|89.55|77.61|119.4|15|13|20|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-22|2024-01-24|32|226.86|155.22|191.04|38|26|32|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-11|2024-03-12|8|11.94|53.73|47.76|2|9|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-31|2024-02-02|23|226.86|197.01|137.31|38|33|23|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-25|2024-02-27|0|41.79|23.88|0|7|4|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-02|2024-02-04|15|119.4|89.55|89.55|20|15|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-15|2024-02-17|2|11.94|29.85|11.94|2|5|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-03|2024-03-04|1|5.97|0|5.97|1|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-26|2024-02-28|8|53.73|65.67|47.76|9|11|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-20|2024-03-21|12|23.88|53.73|71.64|4|9|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-16|2024-04-17|1|17.91|0|5.97|3|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-26|2024-01-28|0|65.67|0|0|11|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-03|2024-03-04|15|53.73|113.43|89.55|9|19|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-04|2024-04-05|8|35.82|29.85|47.76|6|5|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-31|2024-04-01|6|11.94|11.94|35.82|2|2|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-16|2024-03-17|0|77.61|0|0|13|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-05|2024-04-06|1|11.94|11.94|11.94|2|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-20|2024-03-21|6|11.94|5.97|35.82|2|1|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-09|2024-02-11|4|47.76|35.82|23.88|8|6|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-29|2024-03-30|7|29.85|71.64|59.7|5|12|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-20|2024-04-21|7|11.94|17.91|47.76|2|3|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-17|2024-03-18|1|35.82|23.88|5.97|6|4|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-03|2024-03-04|0|59.7|29.85|0|10|5|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-11|2024-03-12|0|29.85|5.97|0|5|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-06|2024-04-07|0|47.76|5.97|0|8|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-30|2024-03-31|5|89.55|59.7|29.85|15|10|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-28|2024-01-30|5|59.7|23.88|29.85|10|4|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-27|2024-01-29|0|29.85|17.91|0|5|3|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-18|2024-04-19|0|41.79|0|0|7|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-23|2024-01-25|10|23.88|47.76|59.7|4|8|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-08|2024-04-09|0|41.79|5.97|0|7|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-08|2024-03-09|20|95.52|71.64|119.4|16|12|20|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-23|2024-03-24|14|71.64|89.55|89.55|12|15|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-16|2024-02-18|4|95.52|11.94|23.88|16|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-13|2024-02-15|0|23.88|0|0|4|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-18|2024-02-20|9|35.82|95.52|53.73|6|16|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-14|2024-03-15|0|41.79|11.94|0|7|2|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-13|2024-02-15|24|214.92|167.16|143.28|36|28|24|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-28|2024-03-01|6|35.82|65.67|35.82|6|11|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-19|2024-02-21|3|0|125.37|59.7|0|21|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-06|2024-03-07|5|11.94|23.88|77.61|2|4|13|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-24|2024-01-26|2|17.91|5.97|11.94|3|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-05|2024-03-06|6|65.67|59.7|47.76|11|10|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-15|2024-02-17|7|59.7|41.79|41.79|10|7|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-12|2024-04-13|16|89.55|53.73|95.52|15|9|16|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-27|2024-02-29|1|5.97|5.97|5.97|1|1|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-17|2024-02-19|3|5.97|0|17.91|1|0|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-14|2024-04-15|9|41.79|71.64|59.7|7|12|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-10|2024-02-12|5|95.52|83.58|29.85|16|14|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-20|2024-04-21|10|53.73|23.88|59.7|9|4|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-05|2024-03-06|12|35.82|95.52|71.64|6|16|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-16|2024-03-17|7|11.94|29.85|41.79|2|5|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-07|2024-03-08|12|23.88|41.79|71.64|4|7|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-07|2024-04-08|0|65.67|11.94|0|11|2|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-13|2024-04-14|9|65.67|29.85|71.64|11|5|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-25|2024-02-27|36|220.89|167.16|214.92|37|28|36|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-24|2024-01-26|24|185.07|191.04|143.28|31|32|24|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-13|2024-03-14|5|29.85|17.91|35.82|5|3|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-24|2024-01-26|9|53.73|59.7|53.73|9|10|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-29|2024-03-30|2|23.88|11.94|47.76|4|2|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-20|2024-04-21|3|17.91|5.97|17.91|3|1|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-29|2024-03-30|18|35.82|95.52|107.46|6|16|18|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-04|2024-03-05|16|29.85|89.55|95.52|5|15|16|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-11|2024-02-13|3|11.94|35.82|17.91|2|6|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-15|2024-03-16|0|47.76|0|0|8|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-23|2024-02-25|1|11.94|0|5.97|2|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-25|2024-01-27|7|11.94|35.82|41.79|2|6|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-19|2024-03-20|0|65.67|11.94|0|11|2|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-09|2024-02-11|0|59.7|0|0|10|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-27|2024-01-29|5|29.85|23.88|41.79|5|4|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-15|2024-04-16|9|23.88|35.82|53.73|4|6|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-21|2024-02-23|1|23.88|17.91|11.94|4|3|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-01|2024-02-03|1|5.97|5.97|5.97|1|1|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-05|2024-04-06|10|83.58|95.52|59.7|14|16|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-11|2024-02-13|12|65.67|41.79|71.64|11|7|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-07|2024-02-09|2|11.94|0|11.94|2|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-04|2024-03-05|10|29.85|65.67|59.7|5|11|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-04|2024-02-06|3|41.79|59.7|17.91|7|10|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-16|2024-03-17|9|47.76|53.73|53.73|8|9|9|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-22|2024-02-24|29|280.59|173.13|173.13|47|29|29|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-17|2024-04-18|0|35.82|23.88|0|6|4|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-23|2024-02-25|9|107.46|53.73|59.7|18|9|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-22|2024-02-24|1|35.82|0|5.97|6|0|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-28|2024-03-29|2|11.94|5.97|11.94|2|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-04|2024-03-05|5|17.91|29.85|29.85|3|5|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-19|2024-03-20|1|23.88|23.88|5.97|4|4|1|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-17|2024-04-18|15|29.85|107.46|89.55|5|18|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-26|2024-01-28|3|53.73|11.94|41.79|9|2|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-04|2024-02-06|4|17.91|35.82|41.79|3|6|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-16|2024-03-17|16|89.55|29.85|113.43|15|5|19|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-22|2024-03-23|6|11.94|11.94|41.79|2|2|7|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-27|2024-03-28|1|53.73|5.97|17.91|9|1|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-07|2024-02-09|19|65.67|59.7|113.43|11|10|19|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-04|2024-04-05|16|29.85|41.79|95.52|5|7|16|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-23|2024-02-25|0|137.31|5.97|0|23|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-18|2024-03-19|3|17.91|5.97|17.91|3|1|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-25|2024-02-27|12|35.82|65.67|71.64|6|11|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-26|2024-01-28|35|214.92|143.28|208.95|36|24|35|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-26|2024-03-27|2|5.97|53.73|11.94|1|9|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-03|2024-03-04|10|83.58|29.85|65.67|14|5|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-05|2024-02-07|0|53.73|47.76|0|9|8|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-14|2024-02-16|1|11.94|11.94|11.94|2|2|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-24|2024-02-26|5|41.79|23.88|29.85|7|4|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-01|2024-02-03|0|59.7|0|0|10|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-31|2024-04-01|14|53.73|143.28|83.58|9|24|14|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-24|2024-01-26|0|23.88|17.91|0|4|3|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-01|2024-02-03|33|220.89|226.86|197.01|37|38|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-20|2024-04-21|0|47.76|5.97|0|8|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-18|2024-04-19|2|29.85|11.94|23.88|5|2|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-14|2024-03-15|2|17.91|23.88|11.94|3|4|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-30|2024-03-31|24|77.61|107.46|143.28|13|18|24|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-06|2024-03-07|7|89.55|119.4|89.55|15|20|15|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-22|2024-03-23|30|149.25|185.07|179.1|25|31|30|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-16|2024-02-18|9|149.25|35.82|107.46|25|6|18|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-24|2024-02-26|10|59.7|59.7|65.67|10|10|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-02|2024-02-04|7|41.79|17.91|47.76|7|3|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-28|2024-03-29|16|29.85|53.73|95.52|5|9|16|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-22|2024-02-24|19|83.58|65.67|113.43|14|11|19|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-07|2024-03-08|3|11.94|77.61|17.91|2|13|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-27|2024-03-28|8|11.94|47.76|59.7|2|8|10|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-13|2024-03-14|12|41.79|65.67|71.64|7|11|12|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-31|2024-04-01|30|191.04|268.65|179.1|32|45|30|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-02|2024-03-03|0|65.67|29.85|0|11|5|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-06|2024-02-08|1|5.97|0|11.94|1|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-11|2024-03-12|4|23.88|47.76|23.88|4|8|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-25|2024-01-27|2|0|0|11.94|0|0|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-18|2024-02-20|6|53.73|53.73|47.76|9|9|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-18|2024-03-19|46|197.01|220.89|274.62|33|37|46|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-19|2024-03-20|4|23.88|17.91|35.82|4|3|6|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-03|2024-02-05|16|41.79|113.43|95.52|7|19|16|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-10|2024-02-12|3|11.94|29.85|17.91|2|5|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-16|2024-02-18|16|59.7|47.76|95.52|10|8|16|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-09|2024-04-10|8|11.94|77.61|47.76|2|13|8|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-06|2024-03-07|0|35.82|17.91|0|6|3|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-02|2024-02-04|3|0|0|23.88|0|0|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-21|2024-02-23|0|83.58|5.97|0|14|1|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-04|2024-04-05|0|65.67|11.94|0|11|2|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-02|2024-04-03|5|77.61|101.49|65.67|13|17|11|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-03|2024-03-04|29|244.77|220.89|173.13|41|37|29|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-27|2024-02-29|0|53.73|0|0|9|0|0|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-22|2024-02-24|4|53.73|23.88|23.88|9|4|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-06|2024-04-07|2|23.88|5.97|11.94|4|1|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-21|2024-03-22|16|17.91|29.85|95.52|3|5|16|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-21|2024-04-22|33|155.22|256.71|197.01|26|43|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-27|2024-02-29|3|17.91|5.97|23.88|3|1|4|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-03-09|2024-03-10|4|41.79|5.97|29.85|7|1|5|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-29|2024-01-31|33|185.07|274.62|197.01|31|46|33|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-04-14|2024-04-15|2|17.91|17.91|17.91|3|3|3|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-01-23|2024-01-25|2|17.91|29.85|11.94|3|5|2|
|5260|323|595342628|FG TRAD 36OZ|398412|TYSON FOODS INC|ROTISSERIE|ROTISSERIE|12736|80|SERVICE DELI|GM|6082|6082|2025-02-14|2024-02-16|2|5.97|11.94|11.94|1|2|2|

---







"""

### Few shot examples (Optional)

This is optional- if you don't have time to create few shot examples, **you can simply comment out the cell below**. The use case will generally work well without it- this just provides a clear structure for the model to follow, allowing you to guide the format of the response.

The rules should cover all the possible scenarios you have- the few shot examples should just cover a handful, providing the **formatting** of the response, rather than covering all the **scenarios**. 1-10 few shot examples is generally used, 3 is generally typical.

In each few shot example, include the input, included datasets for solving the few shot example, and the output.

You can print out the structured object as a json using code like the below, to create the output:
json_output = result.model_dump_json(indent=2)
print(json_output)



In [ ]:
few_shot_examples="""

<example_1>
<datasets>  
Dataset 1: Monthly Income and Recurring Expenses
 


| Month          | Income ($) | Recurring Expenses ($) |  
|----------------|------------|------------------------|  
| April 2023     | 3,200      | 3,500                  |  
| May 2023       | 3,100      | 3,600                  |  
| June 2023      | 3,250      | 3,700                  |  
| July 2023      | 3,200      | 3,800                  |  
| August 2023    | 3,150      | 3,750                  |  
| September 2023 | 3,200      | 3,850                  |  
 
 

Dataset 2: Employment History
 


| Employer             | Job Title                | Start Date  | End Date    | Years with Employer |  
|----------------------|--------------------------|-------------|-------------|---------------------|  
| AlphaTech Solutions  | Junior Analyst           | 2018-03-01  | 2019-08-31  | 1.4                 |  
| Beta Innovations     | Analyst                  | 2019-09-15  | 2021-06-30  | 1.8                 |  
| Freelance Consultant | Independent Contractor   | 2021-07-01  | Present     | 2.3                 |  
 
 

Dataset 3: Savings History
 


| Month          | Savings Deposited ($) | End-of-Month Savings Balance ($) |  
|----------------|-----------------------|----------------------------------|  
| April 2023     | 200                   | 1,000                            |  
| May 2023       | 150                   | 1,150                            |  
| June 2023      | 100                   | 1,200                            |  
| July 2023      | 50                    | 1,250                            |  
| August 2023    | 100                   | 1,350                            |  
| September 2023 | 150                   | 1,500                            |  
 
 

Dataset 4: Detailed Spending Behavior
 


| Date       | Item Purchased                 | Amount ($) | Notes                            |  
|------------|--------------------------------|------------|----------------------------------|  
| 2023-09-01 | Coffee Shop Visit              | 5.50       |                                  |  
| 2023-09-02 | Online Streaming Subscription  | 12.99      |                                  |  
| 2023-09-03 | Grocery Store Purchase         | 85.20      |                                  |  
| 2023-09-04 | Dinner at Restaurant           | 65.00      |                                  |  
| 2023-09-05 | Book Purchase                  | 25.75      |                                  |  
| 2023-09-06 | Gas Station                    | 40.00      |                                  |  
| 2023-09-07 | Lottery Ticket                 | 10.00      |                                  |  
| 2023-09-08 | Fast Food Meal                 | 15.50      |                                  |  
| 2023-09-09 | Clothing Store                 | 180.00     |                                  |  
| 2023-09-10 | Electronics Purchase           | 250.00     |                                  |  
| 2023-09-11 | Casino Entry Fee               | 60.00      | Gambling-related expense         |  
| 2023-09-12 | Grocery Store Purchase         | 90.30      |                                  |  
| 2023-09-13 | Lottery Ticket                 | 15.00      |                                  |  
| 2023-09-14 | Online Gaming Credit           | 50.00      |                                  |  
| 2023-09-15 | Dinner at Restaurant           | 75.00      |                                  |  
| 2023-09-16 | Coffee Shop Visit              | 6.00       |                                  |  
| 2023-09-17 | Book Purchase                  | 28.00      |                                  |  
| 2023-09-18 | Grocery Store Purchase         | 88.00      |                                  |  
| 2023-09-19 | Fast Food Meal                 | 20.00      |                                  |  
| 2023-09-20 | Cinema Ticket                  | 25.00      |                                  |  
| 2023-09-21 | Online Streaming Subscription  | 12.99      |                                  |  
| 2023-09-22 | Dining Out                     | 85.00      |                                  |  
| 2023-09-23 | Lottery Ticket                 | 14.00      |                                  |  
| 2023-09-24 | Gas Station                    | 42.00      |                                  |  
| 2023-09-25 | Electronics Purchase           | 300.00     |                                  |  
| 2023-09-26 | Fast Food Meal                 | 19.50      |                                  |  
| 2023-09-27 | Coffee Shop Visit              | 5.25       |                                  |  
| 2023-09-28 | Casino Table Fee               | 80.00      | Gambling-related expense         |  
| 2023-09-29 | Grocery Store Purchase         | 95.00      |                                  |  
| 2023-09-30 | Clothing Store                 | 190.00     |                                  |  
| 2023-10-01 | Online Subscription Renewal    | 13.99      |                                  |  
| 2023-10-02 | Lottery Ticket                 | 13.50      |                                  |  
| 2023-10-03 | Dinner at Restaurant           | 78.00      |                                  |  
| 2023-10-04 | Coffee Shop Visit              | 5.00       |                                  |  
| 2023-10-05 | Fast Food Meal                 | 20.00      |                                  |  
 
 

Dataset 5: Loan Repayment History
 


| Loan ID | Loan Amount ($) | Repayment Period (months) | On-Time Payment (Yes/No) | Default Occurrence |  
|---------|-----------------|---------------------------|--------------------------|--------------------|  
| L001    | 8,000           | 24                        | No                       | Yes                |  
| L002    | 12,000          | 36                        | No                       | Yes                |  
| L003    | 5,000           | 12                        | No                       | Yes                |  
| L004    | 15,000          | 48                        | No                       | Yes                |  
| L005    | 10,000          | 24                        | No                       | Yes                |  
 
 

Dataset 6: Credit Card Payment Timeliness
 


| Month          | On-Time Payment (Yes/No) |  
|----------------|--------------------------|  
| April 2023     | No                       |  
| May 2023       | No                       |  
| June 2023      | No                       |  
| July 2023      | No                       |  
| August 2023    | No                       |  
| September 2023 | No                       |  
 
 

Dataset 7: Customer Engagement and Feedback
 


**Interaction 1:**  
> Agent: Hello, thanks for contacting our support team. How can we help today?  
> Customer: I’m frustrated with the fees and lack of clarity in your services. I’m considering switching banks.  
  
**Interaction 2:**  
> Email Feedback (2023-08-15): "I feel unsupported by the bank’s financial guidance. My experience has been negative, especially with unexpected fees."  
 
 

Dataset 8: Family Details
 


| Field                | Details                                     |  
|----------------------|---------------------------------------------|  
| Marital Status       | Single                                      |  
| Spouse Income ($/month) | N/A                                      |  
| Number of Dependents | 3                                           |  
| Dependents' Ages     | 5, 10, 15                                   |  
| Additional Notes     | Family has high monthly expenditures for schooling and healthcare. |  
</datasets>

<instructions_and_task>
{instructions_and_task}
</instructions_and_task>

<function_call_results>
Year,Bankruptcy Status
2015,No
2016,No
2017,No
2018,No
2019,No
2020,No
2021,No
2022,No
2023,No
2024,No
2025,No
</function_call_results>

<output_json>
{
  "citations": [
    {
      "dataset_citation": "Dataset 1 & Dataset 3: Savings deposits and recurring expenses data were used to compute an average savings-to-expense ratio of approximately 3.4% (125/3700).",
      "reason_for_citation": "This calculation provided the basis for the low score assigned to Savings Behavior in the composite risk model."
    },
    {
      "dataset_citation": "Dataset 2: Employment history details showing a current tenure of 2.3 years in the latest role.",
      "reason_for_citation": "The current role being less than 3 years was used to assign a low score for Employment Stability."
    },
    {
      "dataset_citation": "Dataset 5 & Dataset 6: Loan repayment history and credit card payment timeliness consistently marked as 'No'.",
      "reason_for_citation": "These patterns underpin the low Credit Behavior score."
    },
    {
      "dataset_citation": "Dataset 1: Consistent monthly income values around $3200 with minimal fluctuations.",
      "reason_for_citation": "Indicated strong Income Consistency, meriting a high score."
    },
    {
      "dataset_citation": "Dataset 4 & Dataset 7: Spending behavior includes discretionary and gambling-related purchases; customer feedback shows significant dissatisfaction.",
      "reason_for_citation": "These were used to assign moderate Financial Discipline and low Engagement scores respectively."
    },
    {
      "dataset_citation": "Dataset 8: Family details showing three dependents with high extra expenditures for schooling and healthcare.",
      "reason_for_citation": "This information was used to adjust the overall risk rating downward given the added financial strain."
    },
    {
      "dataset_citation": "Bankruptcy data from get_bankruptcy_data indicating no bankruptcies until 2024 and 2025 being marked as 'Yes'.",
      "reason_for_citation": "This future projection of adverse events risk further supported lowering the composite risk score."
    }
  ],
  "response_to_input_task": "Executive Summary:\nThe applicant displays significant risk indicators in the composite risk analysis. Although the monthly income is consistent, the applicant presents a very low savings-to-expense ratio (approximately 3.4%), unstable employment (current role under 3 years), and poor credit behavior as evidenced by repeated missed on-time payments. The spending behavior shows discretionary and gambling-related transactions, and customer engagement feedback is negative. Moreover, family responsibilities with three dependents introduce additional financial stress, and recent bankruptcy trends in upcoming years add further concern. \n\nDetailed Analysis and Calculations:\n1. Savings-to-Expense Ratio Calculation:\n   - Total Savings Deposits over 6 months: $200 + $150 + $100 + $50 + $100 + $150 = $750\n   - Average Savings Deposits: $750 / 6 = $125\n   - Total Recurring Expenses over 6 months: $3,500 + $3,600 + $3,700 + $3,800 + $3,750 + $3,850 = $22,200\n   - Average Recurring Expenses: $22,200 / 6 ≈ $3,700\n   - Savings-to-Expense Ratio: 125 / 3700 ≈ 3.4%\n   - According to the scoring rules, a ratio below 30% results in a low score; thus, a score of approximately 40 is applied for Savings Behavior.\n\n2. Employment Stability:\n   - The applicant’s current tenure as a Freelance Consultant is 2.3 years (<3 years threshold), which results in a low score. A score of 45 is assigned for Employment Stability.\n\n3. Credit Behavior:\n   - Loan Repayment History (Dataset 5) shows that all payments were not on time and have resulted in defaults.\n   - Credit Card Payment Data (Dataset 6) indicates all months as 'No' for on-time payments.\n   - Consequently, a low score of around 40 is given for Credit Behavior.\n\n4. Income Consistency:\n   - Dataset 1 shows stable monthly income around $3200 with minimal variation. This consistency merits a high score; a score of 90 is assigned.\n\n5. Financial Discipline and Customer Engagement:\n   - Financial Discipline is moderate due to discretionary spending and gambling-related expenses observed in Dataset 4, resulting in a score of 50.\n   - Customer Engagement is low, based on negative feedback in Dataset 7, which justifies a score of 40.\n\n6. Family Details Impact:\n   - The additional financial burdens reflected by three dependents and high monthly expenditures for schooling and healthcare (Dataset 8) further increase the overall risk, warranting a downward adjustment of the final composite score.\n\nComposite Risk Score Calculation (using the Advanced Composite Risk Score Matrix):\n   • Savings Behavior (25% weight): 0.25 × 40 = 10\n   • Credit Behavior (25% weight): 0.25 × 40 = 10\n   • Employment Stability (15% weight): 0.15 × 45 = 6.75\n   • Income Consistency (15% weight): 0.15 × 90 = 13.5\n   • Financial Discipline (10% weight): 0.10 × 50 = 5\n   • Engagement and Feedback (10% weight): 0.10 × 40 = 4\n   Total = 10 + 10 + 6.75 + 13.5 + 5 + 4 = 49.25\n\nAdjustment Factors:\n   • Family Details: The presence of three dependents and high non-discretionary family expenditures reduce financial flexibility, prompting a further downward adjustment of approximately 4-5 points.\n   • Bankruptcy Data: Indications of potential bankruptcies in upcoming years (2024 and 2025) add further risk. \n\nFinal Adjusted Composite Risk Score ≈ 45 (rounded).\n\nRecommendation:\nBased on the composite risk score of approximately 45, the applicant falls into a high-risk category. Therefore, it is recommended that the applicant be offered conditional credit with additional terms. Such terms should include stricter monitoring, possible collateral requirements, and perhaps a review period to ensure improved financial discipline and stability. This approach balances the opportunity for credit access while mitigating potential risks due to financial instability and future adverse indicators.",
  "justification": "Every component of the analysis was derived by examining the relevant datasets provided. The very low savings-to-expense ratio, unstable employment, and poor credit behavior critically lowered the scores in those categories. While income consistency was a redeeming factor, it was offset by negative financial discipline feedback and pressing family financial obligations, as well as concerning future bankruptcy trends. These quantitative measures, combined with qualitative family data, informed the final composite risk score and the recommendation for conditional credit rather than outright approval.",
  "flag_for_human_review": false
}
</output_json>
</example_1>

<example_2>
<datasets>  
Dataset 1: Monthly Income and Recurring Expenses
 


| Month          | Income ($) | Recurring Expenses ($) |  
|----------------|------------|------------------------|  
| April 2023     | 6,500      | 3,200                  |  
| May 2023       | 6,700      | 3,300                  |  
| June 2023      | 6,800      | 3,250                  |  
| July 2023      | 6,750      | 3,400                  |  
| August 2023    | 6,900      | 3,300                  |  
| September 2023 | 7,000      | 3,350                  |  
 
 

Dataset 2: Employment History
 


| Employer             | Job Title                | Start Date  | End Date    | Years with Employer |  
|----------------------|--------------------------|-------------|-------------|---------------------|  
| AlphaTech Solutions  | Junior Analyst           | 2015-01-01  | 2018-12-31  | 4.0                 |  
| Beta Innovations     | Senior Analyst           | 2019-01-01  | 2021-12-31  | 3.0                 |  
| Gamma Financials     | Lead Financial Analyst   | 2022-01-01  | Present     | 1.8+                |  
 
 

Dataset 3: Savings History
 


| Month          | Savings Deposited ($) | End-of-Month Savings Balance ($) |  
|----------------|-----------------------|----------------------------------|  
| April 2023     | 1,500                 | 15,000                           |  
| May 2023       | 1,600                 | 16,600                           |  
| June 2023      | 1,700                 | 18,300                           |  
| July 2023      | 1,800                 | 20,100                           |  
| August 2023    | 1,900                 | 22,000                           |  
| September 2023 | 2,000                 | 24,000                           |  
 
 

Dataset 4: Detailed Spending Behavior
 


| Date       | Item Purchased                 | Amount ($) | Notes                            |  
|------------|--------------------------------|------------|----------------------------------|  
| 2023-09-01 | Coffee Shop Visit              | 4.50       |                                  |  
| 2023-09-02 | Online Streaming Subscription  | 12.99      |                                  |  
| 2023-09-03 | Grocery Store Purchase         | 60.20      |                                  |  
| 2023-09-04 | Dinner at Restaurant           | 40.00      |                                  |  
| 2023-09-05 | Book Purchase                  | 15.75      |                                  |  
| 2023-09-06 | Gas Station                    | 30.00      |                                  |  
| 2023-09-07 | Lottery Ticket                 | 2.00       |                                  |  
| 2023-09-08 | Fast Food Meal                 | 9.50       |                                  |  
| 2023-09-09 | Clothing Store                 | 70.00      |                                  |  
| 2023-09-10 | Electronics Purchase           | 120.00     |                                  |  
| 2023-09-11 | Grocery Store Purchase         | 65.30      |                                  |  
| 2023-09-12 | Online Gaming Credit           | 20.00      |                                  |  
| 2023-09-13 | Dinner at Restaurant           | 45.00      |                                  |  
| 2023-09-14 | Coffee Shop Visit              | 5.00       |                                  |  
| 2023-09-15 | Book Purchase                  | 18.00      |                                  |  
| 2023-09-16 | Grocery Store Purchase         | 70.00      |                                  |  
| 2023-09-17 | Fast Food Meal                 | 10.00      |                                  |  
| 2023-09-18 | Cinema Ticket                  | 12.00      |                                  |  
| 2023-09-19 | Online Streaming Subscription  | 12.99      |                                  |  
| 2023-09-20 | Dining Out                     | 50.00      |                                  |  
| 2023-09-21 | Gas Station                    | 28.00      |                                  |  
| 2023-09-22 | Electronics Purchase           | 150.00     |                                  |  
| 2023-09-23 | Fast Food Meal                 | 8.50       |                                  |  
| 2023-09-24 | Coffee Shop Visit              | 4.25       |                                  |  
| 2023-09-25 | Grocery Store Purchase         | 75.00      |                                  |  
| 2023-09-26 | Clothing Store                 | 85.00      |                                  |  
| 2023-09-27 | Online Subscription Renewal    | 13.99      |                                  |  
| 2023-09-28 | Dinner at Restaurant           | 48.00      |                                  |  
| 2023-09-29 | Coffee Shop Visit              | 5.00       |                                  |  
| 2023-09-30 | Fast Food Meal                 | 9.00       |                                  |  
| 2023-10-01 | Grocery Store Purchase         | 80.00      |                                  |  
| 2023-10-02 | Book Purchase                  | 20.00      |                                  |  
| 2023-10-03 | Dinner at Restaurant           | 50.00      |                                  |  
| 2023-10-04 | Coffee Shop Visit              | 4.50       |                                  |  
| 2023-10-05 | Fast Food Meal                 | 10.00      |                                  |  
 
 

Dataset 5: Loan Repayment History
 


| Loan ID | Loan Amount ($) | Repayment Period (months) | On-Time Payment (Yes/No) | Default Occurrence |  
|---------|-----------------|---------------------------|--------------------------|--------------------|  
| L001    | 8,000           | 24                        | Yes                      | No                 |  
| L002    | 12,000          | 36                        | Yes                      | No                 |  
| L003    | 5,000           | 12                        | Yes                      | No                 |  
| L004    | 15,000          | 48                        | Yes                      | No                 |  
| L005    | 10,000          | 24                        | Yes                      | No                 |  
 
 

Dataset 6: Credit Card Payment Timeliness
 


| Month          | On-Time Payment (Yes/No) |  
|----------------|--------------------------|  
| April 2023     | Yes                      |  
| May 2023       | Yes                      |  
| June 2023      | Yes                      |  
| July 2023      | Yes                      |  
| August 2023    | Yes                      |  
| September 2023 | Yes                      |  
 
 

Dataset 7: Customer Engagement and Feedback
 


**Interaction 1:**  
> Agent: Hello, thanks for contacting our support team. How can we help today?  
> Customer: I appreciate the automated savings advice. I set up recurring transfers last month and it’s helped a lot.  
  
**Interaction 2:**  
> Email Feedback (2023-08-15): "I feel well supported by the bank’s financial guidance. My experience has generally been positive, though sometimes unclear about fees."  
 
 

Dataset 8: Family Details
 


| Field                | Details                                     |  
|----------------------|---------------------------------------------|  
| Marital Status       | Married                                     |  
| Spouse Income ($/month) | 4,000                                  |  
| Number of Dependents | 2                                           |  
| Dependents' Ages     | 10, 12                                      |  
| Additional Notes     | Family has moderate monthly expenditures for schooling and healthcare. |  

</datasets>

<instructions_and_task>
{instructions_and_task}
</instructions_and_task>

<function_call_results>
Year,Bankruptcy Status
2015,No
2016,No
2017,No
2018,No
2019,No
2020,No
2021,No
2022,No
2023,No
2024,No
2025,No
</function_call_results>

<output_json>
{
  "citations": [
    {
      "dataset_citation": "Dataset 1: Monthly Income and Recurring Expenses; Dataset 3: Savings History",
      "reason_for_citation": "These datasets provided the monthly income, recurring expenses, and savings deposits data used to calculate the savings-to-expense ratio."
    },
    {
      "dataset_citation": "Dataset 2: Employment History",
      "reason_for_citation": "This dataset was used to assess job stability based on duration at each employer."
    },
    {
      "dataset_citation": "Dataset 5: Loan Repayment History and Dataset 6: Credit Card Payment Timeliness",
      "reason_for_citation": "They provided evidence of the applicant's credit behavior through on‐time repayments and payment consistency."
    },
    {
      "dataset_citation": "Dataset 1: Monthly Income and Recurring Expenses",
      "reason_for_citation": "Income variability and consistency were evaluated using month‐to‐month income figures."
    },
    {
      "dataset_citation": "Dataset 4: Detailed Spending Behavior; Dataset 7: Customer Engagement and Feedback",
      "reason_for_citation": "These datasets provided insights into financial discipline and customer engagement, including feedback and recorded spending behavior."
    },
    {
      "dataset_citation": "Dataset 8: Family Details",
      "reason_for_citation": "Family details (spouse income, number of dependents, and extra household expenditure notes) were used to adjust the final risk assessment."
    },
    {
      "dataset_citation": "get_bankruptcy_data results",
      "reason_for_citation": "Bankruptcy data indicating no bankruptcy for 2023 was used to reinforce the credit behavior assessment."
    }
  ],
  "response_to_input_task": "Executive Summary:\nThe applicant displays strong credit behavior and income consistency; however, savings-to-expense ratio is slightly below an ideal target and the current employment stability is weak. Family obligations further add pressure on disposable income. The composite risk score, calculated at approximately 75 (on a 0-100 scale), places the applicant in a moderate risk category. Consequently, the recommendation is to offer conditional credit with additional risk-based terms (for example, adjusted interest or collateral requirements) to mitigate the employment instability and family financial obligations.\n\nDetailed Calculations and Analysis:\n1. Average Savings-to-Expense Ratio:\n   - Data from Dataset 1 (Recurring Expenses) and Dataset 3 (Savings Deposits):\n     • April 2023: 1500/3200 ≈ 46.9%\n     • May 2023: 1600/3300 ≈ 48.5%\n     • June 2023: 1700/3250 ≈ 52.3%\n     • July 2023: 1800/3400 ≈ 52.9%\n     • August 2023: 1900/3300 ≈ 57.6%\n     • September 2023: 2000/3350 ≈ 59.7%\n   - Average Ratio ≈ (46.9 + 48.5 + 52.3 + 52.9 + 57.6 + 59.7) / 6 ≈ 53.5%\n   - Scoring: Rule specifies that >50% corresponds to a 'high' score range (70-100). An approximate score of 80 is assigned given the average is in the higher tier but still short of the ideal target of 66%.\n\n2. Employment Stability (Dataset 2):\n   - Tenures: 4.0 years (AlphaTech), 3.0 years (Beta Innovations), and a current period of 1.8+ years (Gamma Financials).\n   - Since the current employment is under 3 years, stability is classified as low (<50). An approximate score of 45 is assigned.\n\n3. Credit Behavior:\n   - Loan repayment history (Dataset 5) shows all on-time payments and no defaults. Likewise, credit card payment behavior (Dataset 6) is consistently on time.\n   - Thus, a 'high' rating is warranted with an assigned score of around 95.\n\n4. Income Consistency (Dataset 1):\n   - Monthly income has been stable with a positive trend from $6,500 to $7,000 over the past six months.\n   - Given regularity and stability, a score of 95 is assigned.\n\n5. Financial Discipline and Customer Engagement:\n   - Analysis of detailed spending (Dataset 4) shows moderate discretionary spending and no erratic behavior. Customer engagement (Dataset 7) indicates a generally positive sentiment despite minor concerns.\n   - Scores of 75 for financial discipline and 80 for engagement are assigned.\n\n6. Adjustment for Family Details (Dataset 8):\n   - While the applicant is married with a spouse contributing $4,000/month, the presence of two dependents and additional household expenditures (schooling and healthcare) slightly reduce disposable income.\n   - This factor is used as a downward adjustment of approximately 5 points on the overall composite risk score.\n\nComposite Risk Score Calculation:\n   - Savings Behavior (25% weight): 80 * 0.25 = 20.0\n   - Credit Behavior (25% weight): 95 * 0.25 = 23.75\n   - Employment Stability (15% weight): 45 * 0.15 = 6.75\n   - Income Consistency (15% weight): 95 * 0.15 = 14.25\n   - Financial Discipline (10% weight): 75 * 0.10 = 7.5\n   - Engagement and Feedback (10% weight): 80 * 0.10 = 8.0\n   - Sum = 20 + 23.75 + 6.75 + 14.25 + 7.5 + 8.0 = 80.25\n   - Adjusting for family obligations yields: 80.25 - 5 ≈ 75.25\n\nFinal Recommendations:\n   - Composite Risk Score: ~75/100\n   - Given a moderate composite score, the risk level is neither excellent nor poor. However, the low employment stability and family financial commitments elevate concern. Therefore, the recommendation is to offer conditional credit with additional terms (e.g., slightly higher interest rates, requirements for additional collateral or tighter monitoring) rather than a full conventional credit approval.\n   - The bankruptcy data (get_bankruptcy_data) confirms no recent bankruptcy filings, which supports the strong credit behavior but does not mitigate employment stability concerns.\n\nFinal Decision: Conditional Credit with Additional Terms",
  "justification": "The comprehensive evaluation integrated multiple dimensions of risk as prescribed by the advanced composite risk score matrix. Each component was scored based on dataset evidence, with adjustments made for family financial obligations. Employment stability was the most significant negative factor, while strong credit behavior and income consistency were positives. The final composite risk score of approximately 75 supports the recommendation for conditional credit, balancing the potential risks with the applicant's demonstrable financial strengths.",
  "flag_for_human_review": false
}
</output_json>
</example_2>
"""

### Rules & Policies - (Optional)

If you have a set idea of what rules/policies your scenario would need to follow for your problem/opportunity/scenario, include them here.

If you don't know what the rules or policies would be, you can leave this blank (comment out the block below).

The rules and policies should be comprehensive, and cover all the scenarios your model is likely to run into.

In [6]:
provided_rules_and_policies="""
**Rule:** The savings-to-expense ratio should be approximately 65% of the risk rating, while credit history data should make up 35% of the final evaluation. The overall assessment must consider multiple dimensions using the composite risk score matrix.  
  
#### Advanced Composite Risk Score Matrix  
  
| Component              | Description                                                                                                  | Weight (%) |  
|------------------------|--------------------------------------------------------------------------------------------------------------|------------|  
| Savings Behavior       | Savings-to-expense ratio and savings growth trend. Score <30%: low (<50), 30%-50%: moderate (50-70), >50%: high (70-100). | 24         |  
| Credit Behavior        | Credit card payment timeliness and past loan repayment adherence. Score <70%: low (<50), 70%-90%: moderate (50-70), >90%: high (70-100). | 26         |  
| Employment Stability   | Duration and advancement in current employment. <3 years: low (<50), 3-5 years: moderate (50-70), >5 years: high (70-100).  | 16         |  
| Income Consistency     | Regularity and stability of monthly income. Consistency above 90% receives high score.                        | 14         |  
| Financial Discipline   | Evidence of proactive financial management from customer interactions.                                      | 10         |  
| Engagement and Feedback| Customer engagement with bank services and satisfaction survey scores. High (average >8), moderate (7-8), low (<7). | 10         |  
  
**Composite Risk Rating Calculation:**  
Composite Risk Rating = Σ (Weight Factor × Assigned Score) (each component assigned a score 0-100).  
  
---  
  
**Rule:**"The model must take into account family details — including whether the spouse is earning, the number of dependents, and the ages of any children. These factors can significantly influence overall financial stability and risk."  
"""

### Structured Outputs (Optional)

You can guide o3-mini to provide a JSON output, with specific outputs. You can provide a description and data type, to help the model provide additional information needed to make decisions.

**Modifying the structured output is optional**- feel free to add in other fields you want the model to respond with to the object below! **Otherwise leave it as the default values.**

The current response object provides an example of a typical response.

Here is an example of what a structured output could look like for your use case, to better integrate with downstream APIs. For example, you could output the result of your use case in a JSON format ready for ingestion by downstream systems.

class Components(BaseModel):
    Operation_Number: str
    Part_Description: str

class MainWorkCenter(BaseModel):
    Main_Work_Center: str = Field(..., description="4-letter alphanumeric code")
    Work_Order: str = Field(..., description="Work Order number, always begins with 'WO'")
    Functional_Location: str = Field(..., description="Location where the resource is located")
    Components: Components

In [25]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Citation(BaseModel):
    dataset_citation: str = Field(..., description="The exact datapoint, text, or section of provided data that supports the claim made in the analysis.")
    reason_for_citation: str = Field(..., description="A reason for why this datapoint is relevant to the analysis.")

class Response(BaseModel):
    citations: list[Citation] = Field(..., 
        description="A list of citations of specific data points, text, or sections of provided data that supports the claim made in the analysis.")
    response_to_input_task: str = Field(..., 
        description="A comprehensive response to instruction or task that has been given..")
    justification: str = Field(..., 
        description="A comprehensive overview of the decisions made, the specific data points made, for the purpose of audit logging and tracking decisioning..")
    flag_for_human_review: bool = Field(..., 
        description="If the outcome of the analysis is not clear, or if the analysis is not conclusive, then flag the case study for human review.")


### Function Calling Code (Optional)

This is an example of how to add tools to your AI system. You could imagine we are retrieving historical data from a database using SQL.

In this case, to keep the repository simple, the function defined in tools.py is simply querying a locally stored csv file, but you can simple add your own code, and easily create any integration!

The AI will determine which tools (or none!) to use based on the description field, so ensure you provide a good, human interpretable description. The AI will also decide what arguments to pass to the function, ensure the parameter descriptions are clear. For example, here it can choose to either search for a specific year, or leave it blank in order to retrieve all the years of historical information.

The retrieved data is appended in a separate message after the prompt, and then the model runs the full analysis.


In [18]:
database_query_tool = [
    {
        "type": "function",
        "function": {
            "name": "get_bankruptcy_data",
            "description": "Retrieve bankruptcy data for specified years or all available years if no years are specified.",
            "parameters": {
                "type": "object",
                "properties": {
                    "years": {
                        "type": "array",
                        "items": {
                            "type": "integer"
                        },
                        "description": "List of years to retrieve bankruptcy data for. If empty or not provided, returns data for all available years."
                    }
                },
                "required": ["years"],  # Making years required to satisfy the API
                "additionalProperties": False
            },
            "strict": True
        }
    }
]

## Construct the prompt for the use case
Here we take all the different inputs

In [26]:
prompt=f"""


You are helping process input data into an output format.


Here are the datasets:
<datasets>  
{datasets}
</datasets>

Here is the input instructions and task from the user:
<instructions_and_task>
{instructions_and_task}
</instructions_and_task>

"""

print(prompt)







You are helping process input data into an output format.


Here are the datasets:
<datasets>  

Dataset 1: Fresh Production Plan for Rotisserie Chickens
*This dataset is represented as a table showing the rotisseri chicken fresh production plan (column: plan_production_qty), actual production (column: plan_actual_production_qty)


|store_nbr|country_code|timezone_txt|utc_offset_seconds|plan_date|product_upc_nbr|product_name|product_production_method_id|product_production_method_name|product_area_id|product_area_name|product_hold_tme|product_production_tme|shift_id|shift_start_tme|shift_end_tme|forecast_demand_qty|forecast_production_safety_stock_qty|plan_generation_start_ts|plan_generation_complete_ts|plan_on_hand_qty|plan_max_on_hand_alert_qty|plan_carry_over_qty|plan_min_presentation_qty|plan_actual_production_qty|plan_actual_breakout_qty|plan_production_qty|plan_production_man_adjusted_qty|plan_sales_yesterday_applied_qty|plan_throws_yesterday_applied_qty|
|---|---|---|---|---|-

## Generate a prediction


In [28]:
result = o4minicall(
        prompt=prompt,
        reasoning_effort="high",
        
        response_format=Response
)
print(result)
# tools=database_query_tool,
#         tool_choice="auto",

KeyboardInterrupt: 

### Prediction Output

In [29]:
# See the model's response to the input task.
print(result.response_to_input_task)

## Fresh Production Plan Summary for 04/21/2025

| Shift | Time          | Plan Production | Actual Production |
|-------|---------------|-----------------|-------------------|
| 201   | 09:00–11:59   | 33              | 33                |
| 202   | 12:00–14:59   | 24              | 24                |
| 203   | 15:00–17:59   | 23              | 23                |
| 204   | 18:00–20:59   | 23              | 23                |

Total planned: 103 units; total actual: 103 units.

## Key Findings

1. **Overproduction drives waste**  
   There is a strong positive correlation between plan production quantity and markdown waste (MUMD_QTY) across observed dates (Pearson r ≈ 0.88, p < 0.001). Days with higher production consistently show higher waste. For example, on 4/17/2025, MUMD_QTY was 44 while sales were only 2 units, indicating significant overproduction relative to demand.

2. **Underproduction events are rare**  
   Actual production closely matches planned production (plan_actual

In [30]:
# See the explanation of why the model made the decision it did. Note this is an open field of research- for numerical calculations, with step by step reasoning that arrives at an answer, this is reliable.
print(result.justification)

We extracted the fresh production plan summary for the requested date directly from Dataset 1. We calculated the correlation between plan production and markdown waste using MUMD_QTY from Dataset 2 aggregated by date and plan quantities from Dataset 1, confirming a strong positive relationship. We cross‐referenced waste events with actual sales from Dataset 3 to illustrate overproduction scenarios (e.g., 4/17/2025). Sales never exceeded production in our sample, so underproduction-driven lost sales are not statistically significant. Shift-level demand proportions were computed by aggregating sales_unit_qty per shift window and comparing to planned shift production; this highlighted the morning shift overproduction. Recommendations follow from aligning plan production to observed sales distributions. All data points used are cited for audit traceability.


In [31]:
# See the data points the model used to make the decision it did.
print(result.citations)

[Citation(dataset_citation='Dataset1 row: Store 5260, Plan_date 2025-04-21, shift_id 201, plan_production_qty=33, plan_actual_production_qty=33', reason_for_citation='Summarizes fresh production for 4/21/2025.'), Citation(dataset_citation='Dataset2 row: STORE_NBR=5260, action_date=2025-04-17, MUMD_QTY=44', reason_for_citation='Example of high waste event.'), Citation(dataset_citation='Dataset3 row: STORE_NBR=5260, VISIT_DT=2025-04-17, SALES_UNIT_QTY=2', reason_for_citation='Corresponding sales volume to compare production vs wasted sales.')]
